# SIMBA vs modified cosine — **raw parent query + raw candidate MS/MS** (T8, Bac and noBac)

Six injections: `T8_Cilni_Bac_rep1–3.mzML` and `T8_Cilni_noBac_rep1–3.mzML`.

**No MGF spectra are used anywhere in scoring.** The MZmine feature table provides precursor m/z, RT and biological abundance. Each injection supplies its own individual cilnidipine parent MS/MS query (feature 12221), selected closest in retention time to the parent feature coordinate (TIC tie-breaker). Each candidate uses one model-independent selected raw scan. SIMBA and modified cosine score precisely the same raw query–candidate pairs. Pooled results aggregate predictions across injections using the existing method.

The primary graph reports the measured **Top-10 biological-enrichment proxy** (not verified metabolite identifications). This experiment does not assume SIMBA wins: all results are recalculated, including ties and losses. Compare with previous MGF-query analyses as a separate experiment; do not mix scores across different query representations.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
# Set BEFORE importing torch (including torch.utils.data) or any SIMBA modules.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
from simba.config import Config
from simba.preprocessor import Preprocessor
from simba.transformers.encoder import Encoder
from simba.transformers.load_data_encoder import LoadDataEncoder
from simba.analog_discovery.fc_layers_analog_discovery import FcLayerAnalogDiscovery
import simba

In [ ]:
import os, sys, re, copy, json, warnings, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict

ROOT = Path(os.getenv('CHEMPROP_DIR', '/Users/sebas/projects/data/drug_simba_application/chemprop2_cilnidipine')).expanduser()
EDGE_FILE = ROOT/'filtered_pairs.tsv'
FEATURE_FILE = ROOT/'featuretable_reformated.csv'
SAMPLE_META = ROOT/'20240517_Metadata_Injection_time_ChemProp2_9timepoints.txt'
CHECKPOINT =  Path('/Users/sebas/projects/data/nikita_analog_discovery/simba_contrastive_step102k.ckpt')
SIMBA_SOURCE = Path(os.getenv('SIMBA_SOURCE', '/Users/sebas/projects/metabolomics')).expanduser()
COSINE_AS_SIMBA_SMOKE_TEST = False  # True only for downstream cosine identity diagnostic
# In diagnostic mode, keep outputs separate so real SIMBA results are not overwritten.
OUT = ROOT/'simba_results_raw_parent_T8'
OUT.mkdir(parents=True, exist_ok=True)

# Feature ID key must match MZmine feature rows and GNPS2 edges.
FEATURE_ID_COL = 'row ID'           # MZmine default; change if needed
SAMPLE_META_ID_COL = 'filename'    # metadata sample filename column
SAMPLE_META_TIME_COL = 'ATRRIBUTE_SampleType'  # timepoint label (0,1,...8 or T0...T8)
SAMPLE_META_GROUP_COL = 'analysis_group'     # e.g. 'cilnidipine_microbe', 'cilnidipine_abiotic'
MICROBIAL_GROUP = 'cilnidipine_microbe'
ABIOTIC_GROUP = 'cilnidipine_abiotic'
# Sample columns in MZmine table may have suffixes such as ' Peak area'; normalize in mapping function below.
NETWORK_SOURCE_COL = 'CLUSTERID1'; NETWORK_TARGET_COL = 'CLUSTERID2'
PARENT_MZ = 493.1967; PARENT_MZ_TOL_PPM = 10
PARENT_FEATURE_ID = 12221           # set the VERIFIED parent node ID if multiple m/z matches
MAX_SPECTRA = None                 # None = full dataset; any cap is explicitly pilot-only
TOP_K = 10; PLOT_K = 10
MIN_PEAKS = 6; ONLY_POSITIVE = True; REQUIRE_VERIFIED_PROTONATED = False
SIMBA_RANKING = 'casmi_msnlib'  # distance ranking: integer MCES ASC, SED ASC, continuous MCES distance ASC; or 'continuous'
RUN_SIMBA = True
RUN_MODEL_SMOKE_TEST = False  # Optional: test one query/reference forward pass before full retrieval
RUN_MODIFIED_COSINE = True
RUN_BIOLOGY = True
RUN_LEARNED_BASELINES = False
# Uploaded notebook uses 0.1 Da modified cosine tolerance for a baseline.
# Published GNPS2 networking used 0.01 Da precursor/fragment tolerances, cosine >0.7, >=6 matches.
BASELINE_FRAGMENT_TOL_DA = 0.1

for k,p in {'FBMN edges':EDGE_FILE,'feature quantification':FEATURE_FILE,
            'sample metadata':SAMPLE_META,'SIMBA checkpoint':CHECKPOINT}.items():
    print(f'{k:24}: {"FOUND" if p.exists() else "MISSING"}  {p}')
print('Output:',OUT)

In [ ]:
def read_tabular(path):
    if not path.exists(): raise FileNotFoundError(f'Required input missing: {path}')
    return pd.read_csv(path, sep='\t' if path.suffix.lower() in ('.tsv','.txt') else ',', low_memory=False)

def list_schema():
    for name,path in [('edges',EDGE_FILE),('features',FEATURE_FILE),('metadata',SAMPLE_META)]:
        if not path.exists(): continue
        d=read_tabular(path)
        print(name, d.shape, '\ncolumns:', d.columns.tolist()[:100], '\nfirst rows:\n', d.head(3).to_string(index=False)[:2000], '\n')
list_schema()

## 1. Feature-table coordinates and query identity — no MGF loading

The cilnidipine query is **feature 12221**. Its precursor m/z and chromatographic retention time are obtained from the MZmine feature table; **its MS/MS fragment peaks are extracted only from each raw mzML injection**. Neither the query nor the candidates use MGF fragment lists. Raw scans are matched using feature-table precursor and retention time. The query is selected independently of SIMBA and modified cosine scores.

In [ ]:
def key_string(x):
    s=str(x).strip()
    return s[:-2] if re.fullmatch(r'\d+\.0',s) else s

In [ ]:
PARENT_FEATURE_ID=key_string(PARENT_FEATURE_ID)
print('Verified cilnidipine feature ID:', PARENT_FEATURE_ID)
print('Query fragment spectra will be taken from each raw mzML file, not from an MGF.')

## 2. Load the biological-enrichment labels (feature-table T0/T8 abundance)

In [ ]:
def numeric_time(x):
    s=str(x).lower().strip().replace('hours','').replace('hour','').replace('hrs','').replace('hr','').replace(' ','')
    s=re.sub(r'^t(?=\d)', '',s).rstrip('h')
    return pd.to_numeric(s, errors='coerce')

def canonical_sample_name(s):
    s=Path(str(s).strip()).name.lower()
    s=re.sub(r'\s+peak\s+area$', '',s)
    s=re.sub(r'\.(mzml|mzxml|raw)$','',s)
    return s.strip()

intensity=None; sample_index=None; bio_summary=pd.DataFrame()
if RUN_BIOLOGY:
    ft=read_tabular(FEATURE_FILE);sm=read_tabular(SAMPLE_META)
    # Actual ChemProp2 metadata labels, verified against uploaded file.
    # Avoid confusing ATTRIBUTE_time_run (injection timestamp) with experimental timepoint.
    sm=sm.copy()
    sm['analysis_group']=np.where((sm['ATTRIBUTE_Drug']=='Cilnidipine') & (sm['ATTRIBUTE_Bacteria']=='Yes'),
        MICROBIAL_GROUP,np.where((sm['ATTRIBUTE_Drug']=='Cilnidipine') & (sm['ATTRIBUTE_Bacteria']=='No'),ABIOTIC_GROUP,'other'))

    if FEATURE_ID_COL not in ft:raise KeyError(f'{FEATURE_ID_COL} missing in feature table; inspect schema')
    for name in [SAMPLE_META_ID_COL,SAMPLE_META_TIME_COL,SAMPLE_META_GROUP_COL]:
        if name not in sm:raise KeyError(f'{name} missing in metadata; inspect schema')
    ft=ft.copy();ft['feature_id']=ft[FEATURE_ID_COL].map(key_string)
    if ft.feature_id.duplicated().any():raise ValueError('Feature table has duplicate feature IDs')
    sm=sm.copy();sm['group']=sm[SAMPLE_META_GROUP_COL].astype(str).str.strip()
    sm=sm[sm.group.isin([MICROBIAL_GROUP,ABIOTIC_GROUP])].copy()
    if not {MICROBIAL_GROUP,ABIOTIC_GROUP}.issubset(sm.group.unique()):
        raise ValueError('Both drug-exposed microbial and drug-exposed abiotic groups are required; set mappings explicitly.')
    sm['time_h']=sm[SAMPLE_META_TIME_COL].map(numeric_time)
    if sm.time_h.isna().any():raise ValueError('Cannot parse timepoints from sample metadata')
    sm['normalized_name']=sm[SAMPLE_META_ID_COL].map(canonical_sample_name)
    if sm.normalized_name.duplicated().any():raise ValueError('Duplicate normalized sample names in metadata')
    # Avoid substring matching, which can silently join the wrong sample.
    numeric_columns={canonical_sample_name(c):c for c in ft.columns if c not in [FEATURE_ID_COL,'feature_id']}
    if len(numeric_columns)!=len(set(numeric_columns)):raise ValueError('Duplicate normalized column names in feature table')
    absent=set(sm.normalized_name)-set(numeric_columns)
    if absent:raise ValueError(f'Missing sample columns in feature table (examples): {sorted(absent)[:12]}. Check Peak area suffix and file naming.')
    sm['data_column']=sm.normalized_name.map(numeric_columns)
    data=ft.set_index('feature_id')[sm.data_column].apply(pd.to_numeric,errors='coerce')
    data.columns=sm.normalized_name
    if (data.dropna(how='all').values<0).any():raise ValueError('Negative feature abundances are invalid')
    intensity=data;sample_index=sm
    print('Matched samples:',len(sm),'by group/timepoint:\n',sm.groupby(['group','time_h']).size().to_string())
    print('Feature-table rows with matched sample abundances:', len(data), 'features; samples:', len(data.columns))
    assert (sm.groupby('group').time_h.nunique()>=2).all(),'Need >=2 timepoints in both groups'
    sm.to_csv(OUT/'audited_sample_metadata.csv',index=False)

In [ ]:
if intensity is not None:
    biological=[]
    times=sorted(set(sample_index.time_h))
    for fid in intensity.index:
        if fid not in intensity.index:continue
        y=intensity.loc[fid]
        def get_values(group,time):
            names=sample_index.loc[(sample_index.group==group)&(sample_index.time_h==time),'normalized_name']
            return y.reindex(names).to_numpy(dtype=float)
        tr=[np.nanmedian(get_values(MICROBIAL_GROUP,t)) for t in times]
        ct=[np.nanmedian(get_values(ABIOTIC_GROUP,t)) for t in times]
        # Missing values stay NaN: don't transform undetected into measured zero.
        if not np.isfinite(tr).any() or not np.isfinite(ct).any():continue
        end_t=times[-1]
        end_micro=np.nanmedian(get_values(MICROBIAL_GROUP,end_t));end_abio=np.nanmedian(get_values(ABIOTIC_GROUP,end_t))
        base_micro=np.nanmedian(get_values(MICROBIAL_GROUP,times[0]));base_abio=np.nanmedian(get_values(ABIOTIC_GROUP,times[0]))
        # Pseudocount scales with detected nonzero feature intensities; declared, not hidden.
        arr=np.r_[tr,ct];pos=arr[np.isfinite(arr)&(arr>0)]
        pseudo=float(max(1.0,0.01*np.median(pos))) if len(pos) else 1.0
        fold=np.log2((end_micro+pseudo)/(end_abio+pseudo)) if np.isfinite(end_micro) and np.isfinite(end_abio) else np.nan
        tchange=np.log2((end_micro+pseudo)/(base_micro+pseudo)) if np.isfinite(end_micro) and np.isfinite(base_micro) else np.nan
        cchange=np.log2((end_abio+pseudo)/(base_abio+pseudo)) if np.isfinite(end_abio) and np.isfinite(base_abio) else np.nan
        biological.append({'feature_id':fid,'log2_microbe_vs_abiotic_last_time':fold,
            'log2_microbial_time_change':tchange,'log2_abiotic_time_change':cchange,
            'bio_support_proxy':bool(np.isfinite(fold) and np.isfinite(tchange) and np.isfinite(cchange) and
                fold>=1 and (tchange-cchange)>=1),
            'microbial_last_median':end_micro,'abiotic_last_median':end_abio,
            'n_times':len(times),'pseudocount':pseudo})
    bio_summary=pd.DataFrame(biological)
    bio_summary.to_csv(OUT/'biological_controls_summary.csv',index=False)
    print('Features satisfying illustrative enrichment proxy:',int(bio_summary.bio_support_proxy.sum()),'of',len(bio_summary))
    print('Proxy thresholds are exploratory; not an FDR, a compound identity or formal statistical significance test.')

## 3. Initialize SIMBA (raw cilnidipine query embedded later)

The model and checkpoint are unchanged. The query embedding is computed separately for **each injection's raw cilnidipine scan**, using the exact same query-eligibility preprocessing as before.

In [ ]:
RUN_SIMBA=True
if RUN_SIMBA:
    if not CHECKPOINT.exists():
        raise FileNotFoundError(f'SIMBA inference cannot run without trained checkpoint: {CHECKPOINT}')
    if SIMBA_SOURCE.exists(): sys.path.insert(0,str(SIMBA_SOURCE))
    os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK','1')
    import torch
    from torch.utils.data import DataLoader
    from spectrum_utils.spectrum import MsmsSpectrum
    sys.modules['src']=simba
    cfg=Config();cfg.D_MODEL=256;cfg.N_LAYERS=5;cfg.TRANSFORMER_CONTEXT=100
    cfg.USE_FINGERPRINT=0;cfg.use_cosine_distance=True
    if not getattr(cfg,'BATCH_SIZE',None):cfg.BATCH_SIZE=32
    pp=Preprocessor()
    encoder=Encoder(str(CHECKPOINT),D_MODEL=cfg.D_MODEL,N_LAYERS=cfg.N_LAYERS,multitasking=True,config=cfg)
    def embed_simba(sp_list):
        ds=LoadDataEncoder.from_spectrums_to_dataset(sp_list,max_num_peaks=int(cfg.TRANSFORMER_CONTEXT))
        loader=DataLoader(ds,batch_size=int(cfg.BATCH_SIZE),num_workers=0)
        return encoder.get_embeddings(loader)
    print('SIMBA encoder ready; raw parent-query embeddings will be computed per injection.')

## 4. Read six raw mzML injections and match MS2 scans to features

In [ ]:
from pathlib import Path
import re, json, time, warnings
from collections import Counter
from pyteomics import mzml
from scipy.stats import spearmanr
from matchms import Spectrum as MatchmsSpectrum
from matchms.similarity import ModifiedCosine

FULL_RAW_OUT = OUT/'six_raw_injections'
FULL_RAW_OUT.mkdir(parents=True,exist_ok=True)
FULL_MATCH_PPM = 15.0
FULL_MATCH_ABS_DA = 0.015
FULL_MATCH_RT_MIN = 0.25
FULL_PEAK_TOL_DA = 0.02
FULL_BATCH = 256
FULL_STRICT_ALL_FILES = True      # Fail unless all six T8 injections exist
FULL_SCORE_ALL_MATCHED_SCANS = False  # optional expensive secondary best-scan sensitivity
FULL_MAKE_PER_INJECTION_PLOTS = False
FULL_TOP_KS = (5, 10, 20, 50, 100)

assert RUN_SIMBA and RUN_MODIFIED_COSINE and RUN_BIOLOGY
assert not COSINE_AS_SIMBA_SMOKE_TEST, 'Disable diagnostic override to run real SIMBA.'
assert 'embed_simba' in globals()
assert 'ft' in globals() and 'bio_summary' in globals()


all_expected = [ROOT/f'T8_Cilni_{cond}_rep{rep}.mzML'
                for cond in ('Bac', 'noBac') for rep in (1, 2, 3)]
file_inventory = pd.DataFrame([{'filename': p.name, 'condition': ('noBac' if '_noBac_' in p.name else 'Bac'),
                                'exists': p.is_file(),
                                'bytes': p.stat().st_size if p.is_file() else 0}
                               for p in all_expected])
file_inventory.to_csv(FULL_RAW_OUT/'file_inventory.csv', index=False)
missing = file_inventory.loc[~file_inventory.exists, 'filename'].tolist()
print(f'Available {len(all_expected)-len(missing)}/{len(all_expected)} mzML files; missing: {missing}')
if missing and FULL_STRICT_ALL_FILES:
    raise FileNotFoundError(f'Missing {len(missing)} raw mzML files from {ROOT}: {missing}')
files_to_run = [p for p in all_expected if p.is_file()]
if not files_to_run:
    raise FileNotFoundError(f'No T8 cilnidipine raw mzML files found in {ROOT}')
print('Outputs:', FULL_RAW_OUT)

In [ ]:
# Coordinate index from the full aligned feature table;
# every feature with a valid MZmine m/z and retention time.

full_coords = ft[
    [FEATURE_ID_COL, 'row m/z', 'row retention time']
].copy()

full_coords['feature_id'] = full_coords[FEATURE_ID_COL].map(key_string)
full_coords['mz'] = pd.to_numeric(
    full_coords['row m/z'], errors='coerce'
)
full_coords['rt_min'] = pd.to_numeric(
    full_coords['row retention time'], errors='coerce'
)

full_coords = (
    full_coords
    .dropna(subset=['mz', 'rt_min'])

    .drop_duplicates('feature_id')
    .sort_values('mz')
    .reset_index(drop=True)
)

_idx_mass = full_coords.mz.to_numpy(float)
_idx_rt = full_coords.rt_min.to_numpy(float)
_idx_feat = full_coords.feature_id.to_numpy(str)


def _full_rt_min(sp):
    scans = sp.get('scanList', {}).get('scan', [])
    if not scans:
        return np.nan

    x = scans[0].get('scan start time')
    if x is None:
        return np.nan

    try:
        v = float(x)
    except (TypeError, ValueError):
        return np.nan

    unit = str(
        getattr(x, 'unit_info', None)
        or getattr(x, 'unit', None)
        or ''
    ).lower()

    if 'second' in unit or 'uo:0000010' in unit:
        return v / 60.0

    if 'minute' in unit or 'uo:0000031' in unit:
        return v

    if 'hour' in unit or 'uo:0000032' in unit:
        return v * 60.0

    # Unknown unit: do not silently assume minutes.
    return np.nan


def _full_precursor(sp):
    try:
        return float(
            sp['precursorList']['precursor'][0]
              ['selectedIonList']['selectedIon'][0]
              ['selected ion m/z']
        )
    except (KeyError, IndexError, TypeError, ValueError):
        return np.nan


def _full_match(pmz, rt):
    if not np.isfinite(pmz) or not np.isfinite(rt):
        return None, 'missing_coordinate', np.nan, np.nan, 0

    tol = max(
        FULL_MATCH_ABS_DA,
        pmz * FULL_MATCH_PPM * 1e-6
    )

    left = np.searchsorted(_idx_mass, pmz - tol, side='left')
    right = np.searchsorted(_idx_mass, pmz + tol, side='right')

    matches = (
        np.flatnonzero(
            np.abs(_idx_rt[left:right] - rt)
            <= FULL_MATCH_RT_MIN
        ) + left
    )

    if len(matches) != 1:
        status = 'unmatched' if len(matches) == 0 else 'ambiguous'
        return None, status, np.nan, np.nan, len(matches)

    k = int(matches[0])

    return (
        _idx_feat[k],
        'unique',
        (_idx_mass[k] - pmz) / pmz * 1e6,
        _idx_rt[k] - rt,
        1
    )


def _peak_overlap(a, b, tol=FULL_PEAK_TOL_DA):
    a = np.sort(np.asarray(a, float))
    b = np.sort(np.asarray(b, float))

    i = j = count = 0

    while i < len(a) and j < len(b):
        d = a[i] - b[j]

        if abs(d) <= tol:
            count += 1
            i += 1
            j += 1
        elif d < 0:
            i += 1
        else:
            j += 1

    return count


def _read_select_injection(path, keep_all=False):
    best = {}
    all_unique = []
    audit = Counter()
    comparisons = []

    with mzml.MzML(str(path), use_index=False) as reader:
        for sp in reader:

            if int(sp.get('ms level', 0)) != 2:
                continue

            audit['ms2_total'] += 1

            pmz = _full_precursor(sp)
            rt = _full_rt_min(sp)

            m = np.asarray(sp.get('m/z array', []), float)
            y = np.asarray(sp.get('intensity array', []), float)

            valid = (
                np.isfinite(m)
                & np.isfinite(y)
                & (m > 0)
                & (y > 0)
            )

            m = m[valid]
            y = y[valid]

            if (
                len(m) < MIN_PEAKS
                or not np.isfinite(pmz)
                or not np.isfinite(rt)
            ):
                audit['unusable'] += 1
                continue

            fid, status, masserr, rterr, nmatches = (
                _full_match(pmz, rt)
            )

            audit[status] += 1

            if status != 'unique':
                continue

            order = np.argsort(m)
            m = m[order]
            y = y[order]

            # FIX: avoid nested f-string quotation errors.
            scan_number = audit['ms2_total']
            scan_name = sp.get('id') or f'scan_{scan_number}'
            sid = f'{path.stem}::{scan_name}'

            record = {
                'scan_id': sid,
                'feature_id': str(fid),
                'precursor_mz': pmz,
                'rt_min': rt,
                'mz': m,
                'intensity': y,
                'tic': float(y.sum()),
                'rt_error_min': float(rterr),
                'mz_error_ppm': float(masserr)
            }

            if keep_all:
                all_unique.append(record)

            # No MGF spectra are used in the analysis


            # Selection independent of SIMBA and cosine scores.
            # Prefer closest RT; break ties using higher TIC.
            key = (abs(rterr), -record['tic'], sid)

            if fid not in best or key < best[fid][0]:
                best[fid] = (key, record)

    selected = [v[1] for v in best.values()]
    audit['selected_features'] = len(selected)

    return selected, all_unique, audit, comparisons


# Pilot test: inspect the first injection.
trial_selected, trial_all, trial_audit, trial_overlap = (
    _read_select_injection(files_to_run[0], keep_all=False)
)

print('Pilot:', files_to_run[0].name)
print('Audit:', dict(trial_audit))

if trial_audit['ms2_total'] and trial_audit['unique'] == 0:
    raise RuntimeError(
        'No unique scan matches. Check RT units, '
        'feature coordinates and matching tolerances.'
    )

In [ ]:
# Extract a real, individually acquired cilnidipine MS2 spectrum from EACH injection.
# Parent identity is anchored to its row ID and feature-table coordinates.
parent_rows = full_coords.loc[full_coords.feature_id.eq(PARENT_FEATURE_ID)]
if len(parent_rows) != 1:
    raise RuntimeError(f'Parent feature {PARENT_FEATURE_ID} not uniquely present in feature table')
PARENT_RAW_MZ = float(parent_rows.iloc[0].mz)
PARENT_RAW_RT_MIN = float(parent_rows.iloc[0].rt_min)
RAW_QUERY_PPM = 10.0
RAW_QUERY_RT_TOL_MIN = 0.30


def _extract_raw_parent_query(path):
    candidates=[]
    with mzml.MzML(str(path), use_index=False) as reader:
        for scan in reader:
            if int(scan.get('ms level',0)) != 2:
                continue
            pmz=_full_precursor(scan)
            rt=_full_rt_min(scan)
            if not (np.isfinite(pmz) and np.isfinite(rt)):
                continue
            ppm=abs(pmz-PARENT_RAW_MZ)/PARENT_RAW_MZ*1e6
            rt_delta=abs(rt-PARENT_RAW_RT_MIN)
            if ppm>RAW_QUERY_PPM or rt_delta>RAW_QUERY_RT_TOL_MIN:
                continue
            m=np.asarray(scan.get('m/z array',[]),dtype=float)
            y=np.asarray(scan.get('intensity array',[]),dtype=float)
            valid=np.isfinite(m)&np.isfinite(y)&(m>0)&(y>0)
            m=m[valid]; y=y[valid]
            if len(m)<MIN_PEAKS:
                continue
            order=np.argsort(m);m=m[order];y=y[order]
            scan_id=str(scan.get('id','unknown'))
            record={'scan_id':f'{path.stem}::{scan_id}',
                    'feature_id':PARENT_FEATURE_ID, 'precursor_mz':float(pmz),
                    'rt_min':float(rt), 'mz':m, 'intensity':y,
                    'tic':float(y.sum()),
                    'rt_error_min':float(PARENT_RAW_RT_MIN-rt),
                    'mz_error_ppm':float((PARENT_RAW_MZ-pmz)/pmz*1e6),
                    'n_parent_candidates':0}
            # Independent of similarity model: closest feature apex RT, then highest TIC.
            candidates.append((rt_delta,-record['tic'],ppm,record['scan_id'],record))
    if not candidates:
        raise RuntimeError(f'No raw cilnidipine query scan found in {path.name}. '
                           f'Expected m/z={PARENT_RAW_MZ:.6f} ±{RAW_QUERY_PPM}ppm, '
                           f'RT={PARENT_RAW_RT_MIN:.3f}±{RAW_QUERY_RT_TOL_MIN}min. '
                           'Inspect RT units, precursor metadata, and parent coordinates.')
    candidates.sort(key=lambda item:item[:4])
    chosen=candidates[0][4]
    chosen['n_parent_candidates']=len(candidates)
    return chosen


def _query_to_simba(s):
    sp=MsmsSpectrum(str(s['scan_id']),float(s['precursor_mz']),1,
                    s['mz'],s['intensity'])
    sp.params={'precursor_mz':str(s['precursor_mz']),'rt_min':str(s['rt_min'])}
    sp.smiles=''
    return sp

raw_query_by_file={}
raw_query_embedding_by_file={}
raw_query_matchms_by_file={}
query_audit=[]
for path in files_to_run:
    s=_extract_raw_parent_query(path)
    query_sp=_query_to_simba(s)
    preprocessed=pp.preprocess_spectrum(
        copy.deepcopy(query_sp),fragment_tol_mass=10,fragment_tol_mode='ppm',
        min_intensity=0.01,max_num_peaks=1000,scale_intensity=None)
    if preprocessed is None or len(preprocessed.mz)<MIN_PEAKS:
        raise RuntimeError(f'Raw cilnidipine query failed SIMBA preprocessing: {path.name}')
    raw_query_by_file[path.name]=s
    raw_query_embedding_by_file[path.name]=np.asarray(embed_simba([query_sp]))
    raw_query_matchms_by_file[path.name]=MatchmsSpectrum(
        mz=s['mz'],intensities=s['intensity'],
        metadata={'precursor_mz':float(s['precursor_mz'])},
        metadata_harmonization=False)
    query_audit.append({'filename':path.name,'scan_id':s['scan_id'],
                        'precursor_mz':s['precursor_mz'],'rt_min':s['rt_min'],
                        'n_peaks':len(s['mz']), 'parent_candidates':s['n_parent_candidates'],
                        'rt_error_min':s['rt_error_min'],'mz_error_ppm':s['mz_error_ppm']})
raw_query_audit=pd.DataFrame(query_audit)
raw_query_audit.to_csv(FULL_RAW_OUT/'raw_cilnidipine_queries.csv',index=False)
display(raw_query_audit)

# Display the raw query peak lists; all subsequent candidate scoring uses these scans.
fig,axes=plt.subplots(2,3,figsize=(14,6),sharex=True,sharey=True)
for ax,(fname,s) in zip(axes.flat,raw_query_by_file.items()):
    scale=max(float(np.max(s['intensity'])),1.0)
    ax.vlines(s['mz'],0,s['intensity']/scale,lw=0.9)
    ax.set_title(fname.replace('.mzML',''),fontsize=9)
    ax.set_xlabel('fragment m/z')
    ax.set_ylabel('relative intensity')
plt.suptitle('Individual raw cilnidipine query scans (one per injection)')
plt.tight_layout()
plt.savefig(FULL_RAW_OUT/'raw_parent_queries_six_injections.png',dpi=220,bbox_inches='tight')
plt.show()

## 5. Score raw candidate scans against raw cilnidipine query spectra

For each injection, *both* SIMBA and modified cosine use the **same injection's independently selected raw cilnidipine query spectrum**. No MGF fragmentation spectrum is loaded. The query is fixed within an injection, not optimized to favor either scoring method.

In [ ]:
from spectrum_utils.spectrum import MsmsSpectrum
full_metric=ModifiedCosine(tolerance=BASELINE_FRAGMENT_TOL_DA)

def _full_simba_spectrum(s):
    sp=MsmsSpectrum(str(s['scan_id']),float(s['precursor_mz']),1,s['mz'],s['intensity'])
    sp.params={'precursor_mz':str(s['precursor_mz']),'rt_min':str(s['rt_min'])};sp.smiles=''
    return sp

def _full_matchms_spectrum(s):
    return MatchmsSpectrum(mz=s['mz'],intensities=s['intensity'],
        metadata={'precursor_mz':float(s['precursor_mz'])},metadata_harmonization=False)

def _full_cos_score(scan,query_matchms):
    val=full_metric.pair(query_matchms,_full_matchms_spectrum(scan))
    try: return float(val['score'])
    except (TypeError,ValueError,KeyError,IndexError): return float(val[0])

def _score_injection(records,path):
    raw_parent_embedding=raw_query_embedding_by_file[path.name]
    raw_parent_matchms=raw_query_matchms_by_file[path.name]
    eligible=[];qc=[]
    for r in records:
        sp=_full_simba_spectrum(r)
        try:
            test=pp.preprocess_spectrum(copy.deepcopy(sp),fragment_tol_mass=10,fragment_tol_mode='ppm',
                min_intensity=0.01,max_num_peaks=1000,scale_intensity='root')
            passed=test is not None and len(test.mz)>=MIN_PEAKS
        except Exception: passed=False
        if passed and r['feature_id'] != PARENT_FEATURE_ID:
            eligible.append((r,sp))
        else: qc.append({'filename':path.name,'scan_id':r['scan_id'],'feature_id':r['feature_id'],
                        'reason':'raw scan failed preprocessing or is parent query'})
    output=[]
    for start in range(0,len(eligible),FULL_BATCH):
        batch=eligible[start:start+FULL_BATCH]
        emb=np.asarray(embed_simba([sp for _,sp in batch]))
        sed,mces=FcLayerAnalogDiscovery.compute_all_combinations(str(CHECKPOINT),raw_parent_embedding,emb,cfg)
        sed=np.asarray(sed).reshape(len(batch),-1)
        mces=np.asarray(mces).reshape(-1)
        for i,(r,_) in enumerate(batch):
            x=float(mces[i]);d=40-40*x
            output.append({'filename':path.name,'scan_id':r['scan_id'],
                'feature_id':r['feature_id'],'raw_mces':x,'raw_mces_dist':d,
                'raw_integer_mces_dist':int(40-np.round(40*x)),
                'raw_sed_dist_class':int(5-np.argmax(sed[i])),
                'raw_cosine':_full_cos_score(r,raw_parent_matchms),
                'raw_query_scan_id':raw_query_by_file[path.name]['scan_id'],
                'raw_peaks':len(r['mz']),'rt_error_min':r['rt_error_min'],
                'mz_error_ppm':r['mz_error_ppm']})
    return output,qc

_full_scores=[];_full_qc=[];_full_overlap=[];_full_exclusions=[]
for file_i,path in enumerate(files_to_run):
    # Do not retain all raw scans in memory; stream one injection at a time.
    if file_i==0:
        selected,all_scans,qc,overlaps=trial_selected,trial_all,trial_audit,trial_overlap
    else:
        selected,all_scans,qc,overlaps=_read_select_injection(path,keep_all=FULL_SCORE_ALL_MATCHED_SCANS)
    parts=re.fullmatch(r'T(\d+)_Cilni_(Bac|noBac)_rep(\d+)\.mzML',path.name)
    if not parts: raise ValueError(f'Unexpected sample filename: {path.name}')
    t,condition,rep=int(parts[1]),parts[2],int(parts[3])
    chosen=all_scans if FULL_SCORE_ALL_MATCHED_SCANS and all_scans else selected
    rows,exclusions=_score_injection(chosen,path)
    for r in rows:
        r.update(timepoint=t,condition=condition,replicate=rep,
                 scan_strategy='all_scans' if FULL_SCORE_ALL_MATCHED_SCANS else 'nearest_rt')
    _full_scores.extend(rows);_full_exclusions.extend(exclusions)
    _full_qc.append({'filename':path.name,'timepoint':t,'condition':condition,'replicate':rep,
                     **dict(qc),'scored_scans':len(rows),'excluded_after_matching':len(exclusions)})
    print(f'[{file_i+1:02}/{len(files_to_run)}] {path.name}: {len(rows)} scored, {qc["unique"]} uniquely matched MS2 scans')

full_scan_scores=pd.DataFrame(_full_scores)
full_injection_qc=pd.DataFrame(_full_qc).fillna(0)
full_peak_comparison=pd.DataFrame(_full_overlap)
full_exclusions=pd.DataFrame(_full_exclusions,columns=['filename','scan_id','feature_id','reason'])
full_scan_scores.to_csv(FULL_RAW_OUT/'scan_scores.csv',index=False)
full_injection_qc.to_csv(FULL_RAW_OUT/'injection_qc.csv',index=False)
# No raw-vs-MGF candidate comparison
full_exclusions.to_csv(FULL_RAW_OUT/'scan_scoring_exclusions.csv',index=False)
print('Total scored scans:',len(full_scan_scores),'distinct features:',full_scan_scores.feature_id.nunique())
display(full_injection_qc[['filename','ms2_total','unique','ambiguous','unmatched','selected_features','scored_scans']].head(12))
if full_scan_scores.empty:raise RuntimeError('No scored features in any injection; inspect QC and feature ID matching.')

## 6. Raw-only ranking: replicate-wise and pooled across three injections

Each method is evaluated on exactly the same feature IDs in each pool. In pooled results, SIMBA uses a **single real scan prediction MCES/SED prediction triple**, selected by proximity to the median MCES distance; cosine uses median score. There is no cross-condition pooling.

In [ ]:
# Score table may have multiple scans per feature/injection when sensitivity mode is enabled.
full_primary_scans = (full_scan_scores.sort_values(['filename','feature_id','rt_error_min','scan_id'],
                                                   key=lambda col:col.abs() if col.name=='rt_error_min' else col)
                      .drop_duplicates(['filename','feature_id']).copy())
assert not full_primary_scans.duplicated(['filename','feature_id']).any()
full_primary_scans.to_csv(FULL_RAW_OUT/'primary_single_scan_scores.csv',index=False)

# A coherent SIMBA triple is always taken from one actual scan (nearest median MCES distance),
# not formed from per-head minima across different scans. Cosine is a median across injections.
def _score_pool(subset):
    if subset.empty:return pd.DataFrame()
    g=subset.groupby('feature_id',sort=True)
    cosine=g.raw_cosine.median().rename('raw_cosine_median')
    n=g.size().rename('n_scans')
    distances=g.raw_mces_dist.median().rename('median_raw_mces_dist')
    tmp=subset.join(distances,on='feature_id')
    tmp['median_delta']=(tmp.raw_mces_dist-tmp.median_raw_mces_dist).abs()
    exemplar=tmp.sort_values(['feature_id','median_delta','filename','scan_id']).drop_duplicates('feature_id')
    z=exemplar[['feature_id','raw_integer_mces_dist','raw_sed_dist_class','raw_mces_dist','scan_id']].copy()
    z=z.set_index('feature_id').join([cosine,n,distances]).reset_index()
    if len(bio_summary): z=z.merge(bio_summary[['feature_id','bio_support_proxy']],on='feature_id',how='left',validate='one_to_one')
    else: z['bio_support_proxy']=False
    z['bio_support_proxy']=z.bio_support_proxy.fillna(False).astype(bool)
    simba_order = z.sort_values(['raw_integer_mces_dist','raw_sed_dist_class','raw_mces_dist','feature_id']).index
    z['raw_simba_rank'] = pd.Series(np.arange(1,len(z)+1),index=simba_order).reindex(z.index).to_numpy()
    z['raw_cosine_rank']=z.raw_cosine_median.rank(method='first',ascending=False).astype(int)
    return z

def _eval_pool(z,label,metadata):
    comparisons=[];detailed=[]
    if z.empty:return comparisons,detailed
    zz=z.copy(); zz['pool']=label
    detailed.append(zz)
    for method,typ,column in [('SIMBA','raw','raw_simba_rank'),('Modified cosine','raw','raw_cosine_rank')]:
        for k in FULL_TOP_KS:
            k_actual=min(k,len(z))
            top=z.nsmallest(k_actual,column)
            comparisons.append({'pool':label,**metadata,'method':method,'representation':typ,
                'k_requested':k,'k_actual':k_actual,'candidate_pool_size':len(z),
                'enriched_n':int(top.bio_support_proxy.sum()),
                'precision_proxy_at_k':float(top.bio_support_proxy.mean()),
                'top_feature_ids':';'.join(top.feature_id.astype(str))})
    return comparisons,detailed

all_evaluations=[];all_pool_tables=[]
# Per-injection comparisons: each uses its own matched raw-detected feature pool.
for (filename,t,cond,rep),sub in full_primary_scans.groupby(['filename','timepoint','condition','replicate'],sort=True):
    label=f'T{t}_{cond}_rep{rep}'
    z=_score_pool(sub)
    evaluations,pools=_eval_pool(z,label,{'pool_type':'injection','timepoint':t,'condition':cond,'replicate':rep})
    all_evaluations.extend(evaluations);all_pool_tables.extend(pools)

# Pooled comparisons: all available scans per feature are aggregated with a median;
# each method is evaluated on the same raw candidate feature IDs within a pool.
strata = {}
for cond in ('Bac', 'noBac'):
    condition_scans = full_primary_scans.loc[full_primary_scans.condition.eq(cond)]
    if condition_scans.empty:
        continue
    condition_files = [p for p in files_to_run if f'_Cilni_{cond}_' in p.name]
    strata[f'T8_{cond}_all_replicates'] = condition_scans
    observed_in_all = condition_scans.groupby('feature_id').filename.nunique()
    shared_ids = observed_in_all.loc[observed_in_all.eq(len(condition_files))].index
    strata[f'T8_{cond}_three_replicate_intersection'] = condition_scans.loc[
        condition_scans.feature_id.isin(shared_ids)]

for label, sub in strata.items():
    if sub.empty:
        continue
    cond = 'noBac' if label.startswith('T8_noBac') else 'Bac'
    z = _score_pool(sub)
    evaluations, pools = _eval_pool(z, label,
        {'pool_type': 'pooled', 'timepoint': 8, 'condition': cond, 'replicate': np.nan})
    all_evaluations.extend(evaluations)
    all_pool_tables.extend(pools)

full_topk=pd.DataFrame(all_evaluations)
full_pool_ranks=pd.concat(all_pool_tables,ignore_index=True)
full_topk.to_csv(FULL_RAW_OUT/'raw_only_per_injection_and_pooled_topk.csv',index=False)
full_pool_ranks.to_csv(FULL_RAW_OUT/'raw_only_matched_pool_feature_ranks.csv',index=False)
print('Pooled Bac and noBac results (union of observed features, per-feature median aggregation):')
display(full_topk.loc[full_topk.pool.isin(['T8_Bac_all_replicates', 'T8_noBac_all_replicates']),
    ['method','representation','k_requested','enriched_n','precision_proxy_at_k','candidate_pool_size']])

## 7. **Primary figure: Top-10 biological enrichment**

The plotted statistic is the fraction of Top-10 candidates passing the **exploratory abundance-based proxy**. The expected pooled Bac 7 vs 6 from the previous execution is **not hardcoded**; this notebook recomputes values from the six raw injections. The shared-feature analysis may produce a different winner. Figures include all conditions and comparisons, including cases where cosine ties or wins.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
palette={'SIMBA':'#2874A6','Modified cosine':'#E67E22'}
# Replicate-specific Top 10: Bac and noBac separately.
d=full_topk.loc[(full_topk.pool_type=='injection') & (full_topk.k_requested==10)].copy()
fig,axs=plt.subplots(1,2,figsize=(12,4.4),sharey=True)
for ax,cond in zip(axs,('Bac','noBac')):
    sub=d[d.condition==cond]
    xs=np.arange(3)
    for method,off in [('SIMBA',-.18),('Modified cosine',.18)]:
        vals=[]
        for rep in (1,2,3):
            rows=sub[(sub.method==method)&(sub.replicate==rep)]
            vals.append(int(rows.enriched_n.iloc[0]) if len(rows) else np.nan)
        ax.bar(xs+off,vals,width=.34,label=method,color=palette[method])
        for x,y in zip(xs+off,vals):
            if np.isfinite(y):ax.text(x,y+.15,str(int(y)),ha='center',fontsize=9)
    ax.set_xticks(xs,[f'Rep {i}' for i in (1,2,3)])
    ax.set(title=f'T8 {cond}: raw MS/MS',ylabel='Proxy-enriched among top 10',ylim=(0,11))
    ax.legend()
fig.suptitle('SIMBA vs modified cosine | top 10 per injection')
fig.tight_layout()
fig.savefig(FULL_RAW_OUT/'top10_by_injection_raw_only.png',dpi=220,bbox_inches='tight')
plt.show()

# Pooled Bac/noBac Top 10, plus the all-three-replicate intersection.
d=full_topk.loc[(full_topk.pool_type=='pooled')&(full_topk.k_requested==10)].copy()
order=['T8_Bac_all_replicates','T8_Bac_three_replicate_intersection',
       'T8_noBac_all_replicates','T8_noBac_three_replicate_intersection']
d=d[d.pool.isin(order)]
fig,ax=plt.subplots(figsize=(11,4.8))
for method,off in [('SIMBA',-.18),('Modified cosine',.18)]:
    vals=d[d.method==method].set_index('pool').reindex(order)['enriched_n'].to_numpy(dtype=float)
    ax.bar(np.arange(4)+off,vals,width=.34,color=palette[method],label=method)
    for x,y in zip(np.arange(4)+off,vals):
        if np.isfinite(y):ax.text(x,y+.12,str(int(y)),ha='center')
ax.set_xticks(range(4),['Bac: pooled','Bac: all 3','noBac: pooled','noBac: all 3'])
ax.set(ylabel='Proxy-enriched among top 10',ylim=(0,11),title='Top-10 retrieval | individual raw MS/MS, pooled replicates')
ax.legend(); fig.tight_layout()
fig.savefig(FULL_RAW_OUT/'top10_pooled_raw_only.png',dpi=220,bbox_inches='tight')
plt.show()
display(d[['pool','method','k_requested','enriched_n','precision_proxy_at_k','candidate_pool_size']])


## 8. Top-10 feature identities and difference between methods

In [ ]:
pools=['T8_Bac_all_replicates','T8_noBac_all_replicates']
for pool in pools:
    sub=full_pool_ranks.loc[full_pool_ranks.pool==pool].copy()
    if sub.empty: continue
    for method,col in [('SIMBA','raw_simba_rank'),('Modified cosine','raw_cosine_rank')]:
        top=sub.nsmallest(min(10,len(sub)),col).copy()
        top['method']=method
        top['rank']=top[col]
        cols=['pool','method','rank','feature_id','bio_support_proxy','raw_mces_dist','raw_integer_mces_dist','raw_sed_dist_class','raw_cosine_median','n_scans']
        top[cols].to_csv(FULL_RAW_OUT/f'top10_{pool}_{method.replace(" ","_")}.csv',index=False)
        print(f'\n{pool} | {method}')
        display(top[cols])
    top_s=set(sub.nsmallest(10,'raw_simba_rank').feature_id)
    top_c=set(sub.nsmallest(10,'raw_cosine_rank').feature_id)
    print(pool,'Top-10 overlap:',len(top_s&top_c),'SIMBA-only:',sorted(top_s-top_c),'cosine-only:',sorted(top_c-top_s))


## 9. Same-feature Bac vs noBac sensitivity check

An important control: limit ranking to features having eligible raw MS/MS in both conditions, so a different candidate pool cannot explain the ranking gap. **Do not interpret missing MS2 acquisition as compound absence.**

In [ ]:
# Per-condition aggregation; no cross-condition mixing before scoring.
cond_pools = {}
for cond in ('Bac', 'noBac'):
    cond_scans = full_primary_scans.loc[full_primary_scans.condition.eq(cond)]
    cond_pools[cond] = _score_pool(cond_scans)
    print(f'{cond}: {len(cond_pools[cond])} distinct scored features')

bac_ids = set(cond_pools['Bac'].feature_id)
no_ids = set(cond_pools['noBac'].feature_id)
shared_ids = sorted(bac_ids & no_ids)
presence_summary = pd.DataFrame([
    {'category': 'Bac only MS2 acquired', 'features': len(bac_ids - no_ids)},
    {'category': 'Both conditions MS2 acquired', 'features': len(shared_ids)},
    {'category': 'noBac only MS2 acquired', 'features': len(no_ids - bac_ids)},
])
presence_summary.to_csv(FULL_RAW_OUT/'condition_ms2_coverage_summary.csv', index=False)
display(presence_summary)
if not shared_ids:
    raise RuntimeError('No features scored in both conditions; inspect mapping and QC.')

# One row per feature, matched across both conditions. All ranks recalculated on this shared universe.
pair = (cond_pools['Bac'].set_index('feature_id').loc[shared_ids]
        .add_prefix('Bac_').join(
            cond_pools['noBac'].set_index('feature_id').loc[shared_ids].add_prefix('noBac_'),
            how='inner').reset_index())
assert pair.feature_id.is_unique and len(pair) == len(shared_ids)
for cond in ('Bac', 'noBac'):
    # Comparison is always feature-for-feature; recalculate ranks within the shared pool.
    keys = [f'{cond}_raw_integer_mces_dist', f'{cond}_raw_sed_dist_class',
            f'{cond}_raw_mces_dist', 'feature_id']
    ordered = pair.sort_values(keys).index
    pair[f'{cond}_SIMBA_rank_shared'] = pd.Series(
        np.arange(1, len(pair)+1), index=ordered).reindex(pair.index).to_numpy()
    pair[f'{cond}_cosine_rank_shared'] = pair[f'{cond}_raw_cosine_median'].rank(
        method='first', ascending=False).astype(int)

# Compare differences in scores and ranking positions for the *same feature*.
pair['delta_SIMBA_mces_distance_Bac_minus_noBac'] = (
    pair.Bac_raw_mces_dist - pair.noBac_raw_mces_dist)
pair['delta_cosine_Bac_minus_noBac'] = (
    pair.Bac_raw_cosine_median - pair.noBac_raw_cosine_median)
pair['delta_SIMBA_rank_Bac_minus_noBac'] = (
    pair.Bac_SIMBA_rank_shared - pair.noBac_SIMBA_rank_shared)
pair['delta_cosine_rank_Bac_minus_noBac'] = (
    pair.Bac_cosine_rank_shared - pair.noBac_cosine_rank_shared)
# Read abundance label from the pre-existing feature table (same label in both conditions).
pair['bio_support_proxy'] = pair.Bac_bio_support_proxy.astype(bool)
assert pair.bio_support_proxy.eq(pair.noBac_bio_support_proxy).all()
pair.to_csv(FULL_RAW_OUT/'Bac_noBac_shared_feature_scores_and_ranks.csv', index=False)
print('Features with raw MS2 scored in both conditions:', len(pair))
display(pair[['feature_id','bio_support_proxy',
              'Bac_SIMBA_rank_shared','noBac_SIMBA_rank_shared',
              'Bac_cosine_rank_shared','noBac_cosine_rank_shared',
              'delta_SIMBA_mces_distance_Bac_minus_noBac',
              'delta_cosine_Bac_minus_noBac']].head(15))


In [ ]:
# Fair Top-K within the intersection of Bac/noBac scored features.
paired_topk_rows = []
for cond in ('Bac','noBac'):
    for method, rank_col in [('SIMBA', f'{cond}_SIMBA_rank_shared'),
                             ('Modified cosine', f'{cond}_cosine_rank_shared')]:
        for k in FULL_TOP_KS:
            chosen = pair.nsmallest(min(k, len(pair)), rank_col)
            paired_topk_rows.append({
                'condition': cond, 'method': method, 'k': k,
                'k_actual': len(chosen), 'common_pool_n': len(pair),
                'bio_proxy_n': int(chosen.bio_support_proxy.sum()),
                'bio_proxy_fraction': float(chosen.bio_support_proxy.mean()),
                'top_feature_ids': ';'.join(chosen.feature_id.astype(str)),
            })
paired_topk = pd.DataFrame(paired_topk_rows)
paired_topk.to_csv(FULL_RAW_OUT/'Bac_noBac_shared_pool_topk.csv', index=False)
display(paired_topk.loc[paired_topk.k.isin([10,20,50,100]),
    ['condition','method','k','common_pool_n','bio_proxy_n','bio_proxy_fraction']])

fig, axes = plt.subplots(1, 2, figsize=(12,4.3), sharey=True)
for ax, cond in zip(axes, ('Bac','noBac')):
    for method in ('SIMBA','Modified cosine'):
        d = paired_topk.loc[(paired_topk.condition==cond) & (paired_topk.method==method)]
        ax.plot(d.k, d.bio_proxy_fraction, marker='o', label=method)
    ax.set(title=f'T8 {cond} — same {len(pair)} shared features', xlabel='Top-K', ylabel='Biological-support proxy fraction')
    ax.set_ylim(0, 1.05)
    ax.legend()
plt.tight_layout()
plt.savefig(FULL_RAW_OUT/'Bac_noBac_shared_pool_topk.png', dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11,4.3))
for ax, (x,y,title) in zip(axes, [
    ('Bac_raw_mces_dist','noBac_raw_mces_dist','SIMBA MCES predicted distance'),
    ('Bac_raw_cosine_median','noBac_raw_cosine_median','Modified cosine score')]):
    ax.scatter(pair.loc[~pair.bio_support_proxy,x],pair.loc[~pair.bio_support_proxy,y],
               alpha=.3,s=12,label='Not proxy-enriched')
    ax.scatter(pair.loc[pair.bio_support_proxy,x],pair.loc[pair.bio_support_proxy,y],
               alpha=.5,s=14,label='Proxy-enriched')
    lo=min(pair[x].min(),pair[y].min()); hi=max(pair[x].max(),pair[y].max())
    ax.plot([lo,hi],[lo,hi],color='gray',lw=1,linestyle='--')
    ax.set(xlabel='Bac',ylabel='noBac',title=title)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FULL_RAW_OUT/'Bac_noBac_shared_feature_score_scatter.png',dpi=160)
plt.show()


## 10. Final interpretation and audit

In [ ]:
# Main top-10 table for reporting, including comparisons that contradict the hypothesis.
summary = full_topk[full_topk.k_requested.eq(10)].copy()
summary = summary[['pool_type','pool','condition','method','candidate_pool_size','enriched_n','precision_proxy_at_k']]
summary.to_csv(FULL_RAW_OUT/'top10_raw_only_summary.csv',index=False)
display(summary.sort_values(['pool_type','pool','method']))
print('Caveats: biological enrichment is NOT a verified cilnidipine transformation;')
print('only one independently selected raw scan per feature per injection was scored;')
print('parent QUERY and candidates both originate from raw mzML (one query scan per injection).')
print('Outputs:',FULL_RAW_OUT)


---

# 11. Manuscript application: network novelty, experimental controls and raw-MS/MS case studies

**Primary experiment:** individual raw MS/MS query and candidates from the six T8 cilnidipine injections, with the same model-independent scan selection and the same eligible feature pool for each ranking method.

**Definitions and limitations:**
- **D1:** a direct neighbor of parent feature `12221` in the *original* GNPS2 molecular network; **distal:** reachable in two or more original-network steps; **disconnected:** present in another component; **not in graph:** absent from the exported network edges. This is **not** a SIMBA molecular network.
- Biological enrichment remains an exploratory abundance proxy. A more stringent **drug-specific triage** adds DMSO + bacteria controls. Neither criterion establishes molecular identity, a metabolic reaction, or statistical significance.
- Mass differences are differences between **observed precursor m/z**; chemical transformation suggestions require consistent adducts, charge states and exclusion of isotopes/in-source fragments.
- The learned baseline models (**DreaMS, MS2DeepScore, Spec2Vec**) can be included only when their scores are supplied **for the same raw query and same raw candidate features**. MGF-derived baseline scores will not be silently mixed into this experiment.

Figures export as **PDF + SVG + 300-dpi PNG** in the `manuscript_figures` subdirectory. Run all cells above this section first.

In [ ]:
import networkx as nx
from IPython.display import display

FIG_DIR = Path("/Users/sebas/Documents/paper_simba/")
FIG_DIR.mkdir(exist_ok=True, parents=True)
PRIMARY_POOL = 'T8_Bac_all_replicates'
CASE_COUNT = 2
CASE_TOP_N = 40
FRAGMENT_TOL_DA = 0.02
MASS_SHIFT_TOL_DA = 0.025
BASELINE_SCORE_FILES = {
    'DreaMS': ROOT/'dreams_raw_scores.csv',
    'MS2DeepScore': ROOT/'ms2deepscore_raw_scores.csv',
    'Spec2Vec': ROOT/'spec2vec_raw_scores.csv',
}
BASELINE_HIGHER_IS_BETTER = {'DreaMS': True, 'MS2DeepScore': True, 'Spec2Vec': True}
COLORS = {'SIMBA':'#236B9A','Modified cosine':'#D77C28',
          'DreaMS':'#7B61A8','MS2DeepScore':'#198B81','Spec2Vec':'#A65D69'}

def save_figure(fig, stem):
    for ext in ('pdf','svg','png'):
        fig.savefig(FIG_DIR/f'{stem}.{ext}', dpi=300, bbox_inches='tight', facecolor='white')

# Inspect exact original GNPS2 edge column names before constructing the graph.
edge_df=read_tabular(EDGE_FILE)
assert {NETWORK_SOURCE_COL,NETWORK_TARGET_COL}.issubset(edge_df.columns), edge_df.columns.tolist()
edge_df=edge_df.dropna(subset=[NETWORK_SOURCE_COL,NETWORK_TARGET_COL]).copy()
edge_df['src']=edge_df[NETWORK_SOURCE_COL].map(key_string)
edge_df['dst']=edge_df[NETWORK_TARGET_COL].map(key_string)
network=nx.Graph()
for r in edge_df.itertuples(index=False):
    if r.src != r.dst:
        network.add_edge(r.src,r.dst)
if PARENT_FEATURE_ID not in network:
    warnings.warn('Parent feature missing from edge file: all graph statuses will be not_in_graph')
network_hops=nx.single_source_shortest_path_length(network,PARENT_FEATURE_ID) if PARENT_FEATURE_ID in network else {}

def network_status(fid):
    fid=key_string(fid)
    if fid==PARENT_FEATURE_ID:return 'parent'
    if fid not in network:return 'not_in_graph'
    if fid not in network_hops:return 'disconnected'
    return 'D1' if network_hops[fid]==1 else 'distal'

pool_df=full_pool_ranks.loc[full_pool_ranks.pool.eq(PRIMARY_POOL)].copy()
if pool_df.empty:raise RuntimeError(f'Missing pool {PRIMARY_POOL}; run previous cells')
if pool_df.feature_id.duplicated().any():raise ValueError('Duplicate candidate IDs in pooled ranks')
pool_df['feature_id']=pool_df.feature_id.map(key_string)
pool_df['network_status']=pool_df.feature_id.map(network_status)
pool_df['network_hops']=pool_df.feature_id.map(network_hops)
coords=full_coords.set_index('feature_id')[['mz','rt_min']]
pool_df=pool_df.join(coords,on='feature_id',validate='one_to_one')
pool_df['delta_precursor_mz']=pool_df['mz']-float(PARENT_RAW_MZ)
pool_df['abs_delta_mz']=pool_df.delta_precursor_mz.abs()
print('Original GNPS graph:', network.number_of_nodes(),'nodes', network.number_of_edges(),'edges')
print('Pooled raw-scored candidates:',len(pool_df))
display(pool_df.network_status.value_counts(dropna=False).to_frame('candidates'))


## 11A. Attach learned baselines to the **identical raw candidate pool**

Provide one CSV per method in the same directory as the raw `.mzML` files, e.g. `dreams_raw_scores.csv`, with columns `feature_id,score`. These must be scores for **raw cilnidipine query vs raw candidate spectra**, aggregated across the three Bac replicates with the same candidate selection protocol. The notebook intentionally does **not** treat earlier MGF-based baseline predictions as raw predictions. Optional `pool` column, if provided, must equal `T8_Bac_all_replicates` for selected rows. Missing or nonfinite coverage excludes that method, with an explicit warning rather than imputing scores. Add true inference for these models separately if desired.

In [ ]:
method_ranks={'SIMBA':'raw_simba_rank','Modified cosine':'raw_cosine_rank'}
for label,path in BASELINE_SCORE_FILES.items():
    if not path.exists():
        print(f'{label}: no raw score file ({path.name}); shown as NOT EVALUATED')
        continue
    scores=read_tabular(path)
    if 'pool' in scores:
        scores=scores.loc[scores.pool.eq(PRIMARY_POOL)].copy()
    if not {'feature_id','score'}.issubset(scores.columns):
        warnings.warn(f'{label}: expected feature_id,score; skipping');continue
    scores['feature_id']=scores.feature_id.map(key_string)
    if scores.feature_id.duplicated().any():
        warnings.warn(f'{label}: duplicate feature IDs; skipping');continue
    m=scores.set_index('feature_id').score.apply(pd.to_numeric,errors='coerce')
    if not pool_df.feature_id.isin(m.index).all() or not np.isfinite(m.reindex(pool_df.feature_id).to_numpy(float)).all():
        warnings.warn(f'{label}: incomplete coverage of the {len(pool_df)} raw Bac features; skipping');continue
    pool_df[label+'_score']=m.reindex(pool_df.feature_id).to_numpy(float)
    col=label+'_rank'
    pool_df[col]=pool_df[label+'_score'].rank(method='first',ascending=not BASELINE_HIGHER_IS_BETTER[label]).astype(int)
    method_ranks[label]=col

print('Methods valid on identical raw pool:',list(method_ranks))
network_top_rows=[]
for method,rank_col in method_ranks.items():
    for k in (10,20,50,100):
        top=pool_df.nsmallest(min(k,len(pool_df)),rank_col)
        for status in ('D1','distal','disconnected','not_in_graph'):
            network_top_rows.append({'method':method,'k':k,'network_status':status,
                'n':int((top.network_status==status).sum()),
                'enriched_n':int((top.network_status.eq(status)&top.bio_support_proxy).sum())})
network_top=pd.DataFrame(network_top_rows)
network_top.to_csv(FIG_DIR/'network_topk_breakdown.csv',index=False)
pool_df.to_csv(FIG_DIR/'raw_bac_network_and_method_ranks.csv',index=False)

fig,axes=plt.subplots(1,2,figsize=(12.5,4.8),sharey=True)
net_pal={'D1':'#315D85','distal':'#77A3BA','disconnected':'#E1A05A','not_in_graph':'#B9B8BB'}
for ax,k in zip(axes,(10,20)):
    sub=network_top[network_top.k.eq(k)]
    left=np.zeros(len(method_ranks));names=list(method_ranks)
    for status in net_pal:
        counts=np.array([sub.loc[(sub.method==m)&(sub.network_status==status),'n'].sum() for m in names])
        ax.barh(names,counts,left=left,color=net_pal[status],label=status)
        left+=counts
    ax.set(xlabel='Retrieved features',title=f'Top {k} vs original GNPS network',xlim=(0,k))
axes[0].invert_yaxis()
axes[1].legend(loc='upper left',bbox_to_anchor=(1.02,1),frameon=False)
fig.suptitle('Network novelty of raw-MS/MS retrieval (same Bac candidate universe)',fontsize=12)
fig.tight_layout();save_figure(fig,'FigA_network_novelty_top10_top20');plt.show()


## 11B. Network view: original GNPS connectivity (not a learned SIMBA network)

The next panel visualizes the cilnidipine component in the *published GNPS network*, emphasizing the top-10 raw SIMBA candidates. Nodes absent from the original edge export or in other components are **listed separately**, rather than incorrectly drawn as edges to the parent.

In [ ]:
if PARENT_FEATURE_ID in network:
    component=network.subgraph(nx.node_connected_component(network,PARENT_FEATURE_ID)).copy()
    keep=set(component.nodes)
    if len(keep)>220:  # readable local component, with all first-degree neighbors
        keep=set(nx.single_source_shortest_path_length(component,PARENT_FEATURE_ID,cutoff=2))
        component=component.subgraph(keep).copy()
    top10=set(pool_df.nsmallest(10,'raw_simba_rank').feature_id)
    fig,ax=plt.subplots(figsize=(8,6.8))
    pos=nx.spring_layout(component,seed=26,k=0.6/max(1,np.sqrt(len(component))))
    colors=['#E74C3C' if node==PARENT_FEATURE_ID else ('#236B9A' if node in top10 else '#D9DFE5') for node in component]
    sizes=[390 if node==PARENT_FEATURE_ID else (155 if node in top10 else 24) for node in component]
    nx.draw_networkx_edges(component,pos,width=.55,alpha=.3,ax=ax)
    nx.draw_networkx_nodes(component,pos,node_color=colors,node_size=sizes,ax=ax)
    label_ids={node:node for node in component if node==PARENT_FEATURE_ID or node in top10}
    nx.draw_networkx_labels(component,pos,labels=label_ids,font_size=8,ax=ax)
    ax.set_title('Original GNPS parent component | raw SIMBA Top 10 highlighted')
    ax.axis('off');fig.tight_layout();save_figure(fig,'FigA2_original_network_parent_component');plt.show()
else:
    print('Parent is absent from original GNPS edge export; network visualization unavailable')
print('SIMBA Top-10 outside first-degree network:')
display(pool_df.nsmallest(10,'raw_simba_rank').loc[lambda d:d.network_status.ne('D1'),
    ['feature_id','raw_simba_rank','raw_cosine_rank','network_status','delta_precursor_mz','bio_support_proxy']])


## 12. Biological controls: microbial enrichment, time dependence and DMSO background

To reduce false positives from general bacterial metabolism, add a **T8 DMSO + bacteria** abundance control (same feature table). A feature can be labelled `drug_specific_triage` only if it passes the original microbial-vs-abiotic proxy **and** is at least twofold higher than DMSO+Bac at T8 with measurable Bac replicate support. All thresholds are adjustable, exploratory and *not* structural verification.

The time-course panel uses the existing **T0–T8 feature table**. The current raw MS/MS spectra still originate only from T8; appearances at other timepoints are inferred from MS1 peak areas, not confirmed new MS/MS acquisitions.

In [ ]:
# Original biological-control columns from metadata cover Cilni Bac / noBac. DMSO Bac
# measurements are additionally read directly from the original MZmine feature table.
DMSO_T8_NAMES=[f't8_dmso_bac_rep{i}' for i in (1,2,3)]
ft_by_id=ft.set_index('feature_id')
normalized_ft_cols={canonical_sample_name(col):col for col in ft.columns if col not in (FEATURE_ID_COL,'feature_id')}
found_dmso=[name for name in DMSO_T8_NAMES if name in normalized_ft_cols]
if not found_dmso:
    warnings.warn('No T8 DMSO+Bac columns found; drug-specific triage unavailable')
    pool_df['log2_cilni_bac_vs_dmso_bac_T8']=np.nan
    pool_df['drug_specific_triage']=False
else:
    if len(found_dmso)<3:
        warnings.warn(f'Only {len(found_dmso)} DMSO+Bac T8 replicates found; review manually')
    dmso=ft_by_id[[normalized_ft_cols[x] for x in found_dmso]].apply(pd.to_numeric,errors='coerce')
    dmso.columns=found_dmso
    dmso_med=dmso.median(axis=1).rename('dmso_t8_median')
    pool_df=pool_df.join(dmso_med,on='feature_id')
    pool_df=pool_df.merge(bio_summary[['feature_id','microbial_last_median','abiotic_last_median','pseudocount',
        'log2_microbe_vs_abiotic_last_time','log2_microbial_time_change','log2_abiotic_time_change']],
        on='feature_id',how='left',validate='one_to_one')
    pool_df['log2_cilni_bac_vs_dmso_bac_T8']=np.log2(
        (pool_df.microbial_last_median+pool_df.pseudocount)/
        (pool_df.dmso_t8_median+pool_df.pseudocount))
    pool_df['drug_specific_triage']=(pool_df.bio_support_proxy &
        pool_df.log2_cilni_bac_vs_dmso_bac_T8.ge(1) &
        pool_df.microbial_last_median.gt(0) &
        np.isfinite(pool_df.log2_cilni_bac_vs_dmso_bac_T8)).fillna(False)
    # Replicate detection check: positive peak area in >= 2/3 T8 Bac injections
    bac_t8_names=sample_index.loc[(sample_index.group==MICROBIAL_GROUP)&(sample_index.time_h.eq(8)), 'normalized_name'].tolist()
    bac_detect=(intensity[bac_t8_names].gt(0).sum(axis=1)).rename('n_positive_Bac_T8')
    pool_df=pool_df.join(bac_detect,on='feature_id')
    pool_df['drug_specific_triage'] &= pool_df.n_positive_Bac_T8.ge(2)

# Rank and quantification provenance: scores are T8 individual raw MS2; fold changes are MZmine MS1.
pool_df.to_csv(FIG_DIR/'raw_bac_candidates_with_biological_controls.csv',index=False)
print('Original proxy positives:',pool_df.bio_support_proxy.sum(),
      'Drug-specific triage positives:',pool_df.drug_specific_triage.sum())
fig,axs=plt.subplots(1,2,figsize=(12.4,4.4))
for ax,k in zip(axs,(10,20)):
    x=np.arange(len(method_ranks)); width=.32
    names=list(method_ranks)
    top=[pool_df.nsmallest(k,col) for col in method_ranks.values()]
    bio=[int(t.bio_support_proxy.sum()) for t in top]
    specific=[int(t.drug_specific_triage.sum()) for t in top]
    ax.bar(x-width/2,bio,width,label='Bac/abiotic proxy',color='#A7BECF')
    ax.bar(x+width/2,specific,width,label='Also enriched vs DMSO+Bac',color='#236B9A')
    for xx,yy in zip(x-width/2,bio):ax.text(xx,yy+.1,str(yy),ha='center',fontsize=9)
    for xx,yy in zip(x+width/2,specific):ax.text(xx,yy+.1,str(yy),ha='center',fontsize=9)
    ax.set(xticks=x,xticklabels=names,ylabel='Number of features',title=f'Top {k}: experimental abundance evidence',ylim=(0,k+1))
    ax.tick_params(axis='x',rotation=20)
axs[0].legend(frameon=False,loc='upper right')
fig.tight_layout();save_figure(fig,'FigB_topk_control_enrichment');plt.show()


## 13. Select two to three *putative* SIMBA examples for expert inspection

The shortlisting is **not manual confirmation**. It prioritizes raw SIMBA Top-40 candidates that are outside D1, pass the drug-specific abundance triage, and have plausible precursor differences and fragment overlap; if fewer than three qualify, it fills from weaker tiers **and labels the evidence tier**. The table contains an editable `expert_annotation` field for reporting only after manual review.

Common mass-shift possibilities below are *neutral-mass approximations*. They should not be presented as chemical transformations without adduct, isotope, charge-state and MS/MS verification.

In [ ]:
MASS_HINTS={
    'H2 addition / reduction':2.01565,
    'H2 loss / oxidation':-2.01565,
    'O addition / hydroxylation':15.99491,
    'CH2 addition / methylation':14.01565,
    'H2O loss / dehydration':-18.01056,
    'H2O addition / hydration':18.01056,
    'NO2 to NH2 (net)':-29.97418,
    'CO loss':-27.99491,
}
def mass_hint(delta,tol=MASS_SHIFT_TOL_DA):
    if not np.isfinite(delta):return 'unknown'
    close=sorted((abs(delta-m),label) for label,m in MASS_HINTS.items())
    return close[0][1] if close and close[0][0]<=tol else 'no simple shift assigned'

# Build selected original raw records for Bac injections (not MGF).
# Only three injections are reopened; no model scoring or arbitrary best-score selection.
raw_bac_records={}
for path in [p for p in files_to_run if '_Cilni_Bac_' in p.name]:
    chosen,_,audit,_=_read_select_injection(path,keep_all=False)
    for record in chosen:
        raw_bac_records[(path.name,str(record['feature_id']))]=record

# Choose the same actual representative scan used in pooled SIMBA ranking where possible.
scan_info=full_primary_scans.loc[full_primary_scans.condition.eq('Bac'),
    ['filename','feature_id','scan_id']].copy()
chosen_records={}
for row in pool_df.itertuples(index=False):
    possible=scan_info.loc[(scan_info.feature_id.astype(str)==row.feature_id)&(scan_info.scan_id==row.scan_id)]
    if len(possible):
        fname=str(possible.iloc[0].filename)
        rec=raw_bac_records.get((fname,row.feature_id))
        if rec is not None:chosen_records[row.feature_id]=(fname,rec)
    if row.feature_id not in chosen_records:
        possible=scan_info.loc[scan_info.feature_id.astype(str)==row.feature_id]
        for rr in possible.itertuples(index=False):
            rec=raw_bac_records.get((rr.filename,row.feature_id))
            if rec is not None:chosen_records[row.feature_id]=(rr.filename,rec);break

def matched_fragments(x,y,tol=FRAGMENT_TOL_DA):
    a=np.sort(np.asarray(x,float));b=np.sort(np.asarray(y,float));i=j=0;pairs=[]
    while i<len(a) and j<len(b):
        if abs(a[i]-b[j])<=tol:pairs.append((float(a[i]),float(b[j])));i+=1;j+=1
        elif a[i]<b[j]:i+=1
        else:j+=1
    return pairs

fragment_rows=[]
for fid,(fname,rec) in chosen_records.items():
    query=raw_query_by_file[fname]
    pairs=matched_fragments(query['mz'],rec['mz'])
    fragment_rows.append({'feature_id':fid,'fragment_matches_0p02Da':len(pairs),
        'raw_peak_coverage':len(pairs)/max(1,len(rec['mz'])),
        'query_peak_coverage':len(pairs)/max(1,len(query['mz'])),
        'raw_scan_file':fname,'raw_scan_id':rec['scan_id']})
fragment_df=pd.DataFrame(fragment_rows)
pool_df=pool_df.merge(fragment_df,on='feature_id',how='left',validate='one_to_one')
pool_df['transformation_hint_unverified']=pool_df.delta_precursor_mz.map(mass_hint)
pool_df['novel_relative_to_D1']=~pool_df.network_status.isin(['D1','parent'])
pool_df['plausible_mass_window_100Da']=pool_df.abs_delta_mz.le(100)
pool_df['ranked_candidate_tier']=np.select([
    pool_df.drug_specific_triage & pool_df.novel_relative_to_D1 & pool_df.plausible_mass_window_100Da & pool_df.fragment_matches_0p02Da.ge(3),
    pool_df.bio_support_proxy & pool_df.novel_relative_to_D1 & pool_df.plausible_mass_window_100Da,
    pool_df.bio_support_proxy & pool_df.novel_relative_to_D1],
    ['drug-associated + fragment support (exploratory)', 'microbial + mass (exploratory)',
     'microbial + network-novel (exploratory)'],default='lower evidence')
priority={'drug-associated + fragment support (exploratory)':0,
          'microbial + mass (exploratory)':1,'microbial + network-novel (exploratory)':2,'lower evidence':3}
shortlist=(pool_df.nsmallest(min(CASE_TOP_N,len(pool_df)),'raw_simba_rank').copy())
shortlist['tier_order']=shortlist.ranked_candidate_tier.map(priority)
shortlist=shortlist.sort_values(['tier_order','raw_simba_rank','abs_delta_mz'])
case_candidates=shortlist.head(CASE_COUNT).copy()
case_candidates['expert_annotation']='NOT REVIEWED — inspect adduct/isotope/structures manually'
case_candidates['confirmed_transformation']=False
cols=['feature_id','raw_simba_rank','raw_cosine_rank','network_status','mz','delta_precursor_mz',
      'raw_integer_mces_dist','raw_sed_dist_class','raw_mces_dist',
      'raw_cosine_median','fragment_matches_0p02Da','bio_support_proxy','drug_specific_triage',
      'log2_cilni_bac_vs_dmso_bac_T8','transformation_hint_unverified',
      'ranked_candidate_tier','raw_scan_file','raw_scan_id','expert_annotation','confirmed_transformation']
case_candidates[cols].to_csv(FIG_DIR/'two_candidate_manual_review_template.csv',index=False)
shortlist.to_csv(FIG_DIR/'candidate_shortlist_all_evidence.csv',index=False)
display(case_candidates[cols])
print('IMPORTANT: transformation hints are mass-shift hypotheses, NOT verified reaction annotations.')


---
# 18. **New raw-MS/MS molecular network using modified cosine**

**Purpose:** The previous molecular-network analysis uses GNPS edges calculated from feature-level MGF spectra. Here we reconstruct an **independent molecular network from individual, non-consensus MS/MS spectra** and determine how far SIMBA- and cosine-retrieved features lie from the raw cilnidipine query in that new network.

**Network nodes:** exactly the raw-MS/MS candidate feature IDs in the pooled T8 + bacteria ranking (`pool_df`), **plus the raw cilnidipine query**. Isolated nodes are deliberately retained. This avoids declaring a candidate `not_in_graph` just because GNPS originally omitted it.

**One raw spectrum per feature:** use the already extracted, preselected raw MS/MS spectra; if multiple T8 bacterial injections contain a feature, select a spectrum from the lowest numbered available replicate. This is independent of either method's similarity score. Use the same representative cilnidipine MS/MS query from T8 Bac rep1 (or the lowest available numbered bacterial injection).

**Edge criterion (GNPS-like, but not identical to GNPS):** modified cosine computed between every pair of raw spectra, default fragment tolerance 0.01 Da, score ≥ 0.7 and at least 6 matched peaks. Precursor-window and local top-peak filtering are optionally enabled. The default graph additionally requires reciprocal top-10 neighbors; an unpruned threshold graph is retained as a sensitivity analysis. Exact equivalence to GNPS requires reproducing its full preprocessing, thresholding and network filtering implementation.

The network is **defined by modified cosine**, so high cosine-to-parent candidates are expected to be direct neighbors; therefore, a larger number of SIMBA candidates at D2+ or disconnected status measures **retrieval complementarity**, not independent proof of better chemical accuracy.

This section may require substantial computation for ~1,000 spectra (~500,000 unordered pairs). Completed edge calculations are cached; reruns reuse the cache when configuration and selected scans match.

In [ ]:
import hashlib
import itertools
import time
from collections import Counter

# Network settings. Do NOT reuse the looser 0.1-Da retrieval-baseline tolerance blindly.
RAW_NET_FRAGMENT_TOL_DA = 0.01
RAW_NET_MIN_MATCHES = 6
RAW_NET_MIN_COSINE = 0.70
RAW_NET_MUTUAL_TOP_N = 10            # GNPS-like reciprocal neighbor pruning
RAW_NET_PRECURSOR_EXCLUSION_DA = 17.0
RAW_NET_APPLY_PRECURSOR_FILTER = True
RAW_NET_APPLY_LOCAL_TOP_PEAKS = True
RAW_NET_LOCAL_WINDOW_DA = 50.0
RAW_NET_PEAKS_PER_WINDOW = 6
RAW_NET_PLOT_KS = (5, 10, 20, 50, 100)
RAW_NET_MAX_CANDIDATES = None        # None = FULL candidate set; a smaller value is strictly a pilot
RAW_NET_FORCE_RECOMPUTE = False
RAW_NET_PROGRESS_EVERY = 25000
RAW_NET_DIR = Path('/Users/sebas/Documents/paper_simba/')
RAW_NET_DIR.mkdir(parents=True, exist_ok=True)

# This section depends on the actual raw selected scans from Section 13.
needed_vars = ('pool_df', 'chosen_records', 'raw_query_by_file', 'full_primary_scans', 'PARENT_FEATURE_ID')
missing_vars = [name for name in needed_vars if name not in globals()]
if missing_vars:
    raise RuntimeError('Run the earlier raw-MS/MS scoring and manuscript cells first: ' + ', '.join(missing_vars))

# Stable representative: earliest numbered injection, not the highest-scoring scan.
def _raw_net_rep_number(filename):
    mm = re.search(r'_rep(\d+)\.mzML$', str(filename))
    return int(mm.group(1)) if mm else 999999

raw_net_pool = pool_df.copy()
if RAW_NET_MAX_CANDIDATES is not None:
    raw_net_pool = raw_net_pool.nsmallest(int(RAW_NET_MAX_CANDIDATES), 'raw_simba_rank').copy()
    print('PILOT ONLY: full-pool conclusions are invalid with candidate cap.')

raw_net_ids = set(raw_net_pool.feature_id.astype(str))
raw_net_candidates = {}
raw_net_missing = []
for fid in sorted(raw_net_ids):
    # Existing chosen_records map uses model-independent nearest-RT raw scans.
    matches = [(f, r) for (f, rr_fid), r in raw_bac_records.items() if rr_fid == fid]
    if not matches:
        raw_net_missing.append(fid)
        continue
    f, rec = min(matches, key=lambda z: (_raw_net_rep_number(z[0]), z[0]))
    raw_net_candidates[fid] = dict(rec)

if raw_net_missing:
    raise RuntimeError(f'{len(raw_net_missing)} pooled candidate(s) lack a selected raw Bac MS/MS scan: {raw_net_missing[:10]}')

raw_net_bac_files = sorted(
    (fn for fn in raw_query_by_file if re.fullmatch(r'T8_Cilni_Bac_rep\d+\.mzML', fn)),
    key=lambda fn: (_raw_net_rep_number(fn), fn)
)
if not raw_net_bac_files:
    raise RuntimeError('No raw parent query for a T8 Bac replicate. Cannot construct raw network.')
raw_net_parent_source = raw_net_bac_files[0]
raw_net_parent_record = dict(raw_query_by_file[raw_net_parent_source])
raw_net_parent_id = str(PARENT_FEATURE_ID)
if raw_net_parent_id in raw_net_candidates:
    raise ValueError('Parent feature erroneously included in the candidate universe')

raw_net_records = {**raw_net_candidates, raw_net_parent_id: raw_net_parent_record}
print('Raw network candidate nodes:', len(raw_net_candidates), 'plus parent query')
print('Parent raw query source:', raw_net_parent_source)
print('Network candidate IDs equal to ranking pool:', set(raw_net_candidates) == raw_net_ids)


## 18A. Consistent raw-MS/MS preparation and an all-pairs modified-cosine network

No consensus/feature-level MGF spectra are used. Precursor filtering and local peak selection apply **equally to every raw spectrum**. To avoid inadvertent overclaiming, this is called a *GNPS-like raw modified-cosine network*, rather than an exact rerun of the GNPS2 workflow.

The all-pairs calculation is cached to CSV together with a configuration fingerprint. The cached threshold-passing edges can be reused when exploring reciprocal top-neighbor thresholds, but the match tolerance or peak processing changes require recomputing pairwise scores.

In [ ]:
from matchms.similarity import ModifiedCosine as RawNetworkModifiedCosine
from matchms import Spectrum as RawNetworkSpectrum


def _raw_network_prepare(rec):
    mz = np.asarray(rec['mz'], dtype=np.float64)
    intens = np.asarray(rec['intensity'], dtype=np.float64)
    good = np.isfinite(mz) & np.isfinite(intens) & (mz > 0) & (intens > 0)
    mz, intens = mz[good], intens[good]
    pmz = float(rec['precursor_mz'])
    if RAW_NET_APPLY_PRECURSOR_FILTER:
        keep = mz < pmz - RAW_NET_PRECURSOR_EXCLUSION_DA
        mz, intens = mz[keep], intens[keep]
    if RAW_NET_APPLY_LOCAL_TOP_PEAKS and len(mz):
        # Retain a peak only if it is among the highest N within its centered 50-Da window.
        order = np.argsort(mz)
        mz, intens = mz[order], intens[order]
        kept = np.zeros(len(mz), dtype=bool)
        half = RAW_NET_LOCAL_WINDOW_DA / 2
        for idx, mass in enumerate(mz):
            lo = np.searchsorted(mz, mass - half, side='left')
            hi = np.searchsorted(mz, mass + half, side='right')
            stronger = np.count_nonzero(intens[lo:hi] > intens[idx])
            equal_before = np.count_nonzero(intens[lo:idx] == intens[idx])
            kept[idx] = stronger + equal_before < RAW_NET_PEAKS_PER_WINDOW
        mz, intens = mz[kept], intens[kept]
    if len(mz):
        order = np.argsort(mz)
        mz, intens = mz[order], intens[order]
    if len(mz) < RAW_NET_MIN_MATCHES:
        return None
    return RawNetworkSpectrum(mz=mz, intensities=intens,
        metadata={'precursor_mz': pmz}, metadata_harmonization=False)

raw_net_spectra = {fid: _raw_network_prepare(rec) for fid, rec in raw_net_records.items()}
raw_net_usable_ids = sorted(fid for fid, sp in raw_net_spectra.items() if sp is not None)
print('Usable raw spectra after network peak filtering:', len(raw_net_usable_ids), '/', len(raw_net_records))
if raw_net_parent_id not in raw_net_usable_ids:
    raise RuntimeError('The raw parent query has too few peaks after filtering. Relax filter settings.')

# Hash both the configuration and the selected actual peak lists; never silently
# reuse a cache corresponding to different spectra or tolerances.
_cfg = {'tol_da':RAW_NET_FRAGMENT_TOL_DA, 'min_matches':RAW_NET_MIN_MATCHES,
        'min_cosine':RAW_NET_MIN_COSINE,
        'precursor_filter':RAW_NET_APPLY_PRECURSOR_FILTER,
        'precursor_exclusion_da':RAW_NET_PRECURSOR_EXCLUSION_DA,
        'local_filter':RAW_NET_APPLY_LOCAL_TOP_PEAKS,
        'window_da':RAW_NET_LOCAL_WINDOW_DA,
        'peaks_per_window':RAW_NET_PEAKS_PER_WINDOW}
_hasher=hashlib.sha256(json.dumps(_cfg, sort_keys=True).encode())
for fid in sorted(raw_net_records):
    rec = raw_net_records[fid]
    _hasher.update(fid.encode())
    _hasher.update(str(rec['scan_id']).encode())
    _hasher.update(np.asarray(rec['mz'], dtype='<f8').tobytes())
    _hasher.update(np.asarray(rec['intensity'], dtype='<f8').tobytes())
raw_net_fingerprint=_hasher.hexdigest()[:20]
raw_net_cache_path=RAW_NET_DIR / f'pairwise_edges_{raw_net_fingerprint}.csv'

if raw_net_cache_path.exists() and not RAW_NET_FORCE_RECOMPUTE:
    raw_net_candidate_edges=pd.read_csv(raw_net_cache_path, dtype={'feature_a':str,'feature_b':str})
    print('Loaded raw-network cached threshold edges:', raw_net_cache_path.name)
else:
    raw_net_metric=RawNetworkModifiedCosine(tolerance=RAW_NET_FRAGMENT_TOL_DA)
    raw_net_edge_rows=[]
    n_pairs=len(raw_net_usable_ids)*(len(raw_net_usable_ids)-1)//2
    start_time=time.monotonic()
    for pair_idx, (a,b) in enumerate(itertools.combinations(raw_net_usable_ids,2),start=1):
        result=raw_net_metric.pair(raw_net_spectra[a],raw_net_spectra[b])
        try:
            value=float(result['score'])
            n_matches=int(result['matches'])
        except (TypeError, KeyError, IndexError, ValueError):
            value=float(result[0]); n_matches=int(result[1])
        if np.isfinite(value) and value >= RAW_NET_MIN_COSINE and n_matches >= RAW_NET_MIN_MATCHES:
            raw_net_edge_rows.append({'feature_a':a,'feature_b':b,
                                      'cosine':value,'matched_peaks':n_matches})
        if pair_idx % RAW_NET_PROGRESS_EVERY == 0:
            print(f'Raw pairwise modified cosine: {pair_idx:,}/{n_pairs:,} pairs; '
                  f'{len(raw_net_edge_rows):,} qualifying edges; '
                  f'{(time.monotonic()-start_time)/60:.1f} minutes elapsed')
    raw_net_candidate_edges=pd.DataFrame(raw_net_edge_rows,
        columns=['feature_a','feature_b','cosine','matched_peaks'])
    raw_net_candidate_edges.to_csv(raw_net_cache_path,index=False)
    print('Saved threshold-edge cache:', raw_net_cache_path.name)

print('Pairs above raw cosine + matched-peak thresholds:',len(raw_net_candidate_edges))
raw_net_candidate_edges.to_csv(RAW_NET_DIR/'raw_modified_cosine_all_threshold_edges.csv',index=False)


## 18B. Construct the graph and classify distance from cilnidipine

- `D1`: directly connected to cilnidipine by a passing raw modified-cosine edge.
- `D2`, `D3`, `D4+`: shortest-path distance from cilnidipine in the raw network.
- `Disconnected`: valid candidate node with no path to cilnidipine, **including isolated nodes**.
- `Insufficient peaks`: candidate present in the shared retrieval pool but with fewer than 6 usable peaks after optional network filtering (shown separately, never counted as disconnected).

Results are shown both **with** reciprocal top-10 neighbor pruning and **without** pruning.

In [ ]:
def _make_raw_graph(edges, all_ids, mutual_n=None):
    g=nx.Graph()
    g.add_nodes_from(all_ids)  # Important: keep isolated candidate nodes!
    ed=edges.copy()
    if mutual_n is not None and not ed.empty:
        # Stable tie-break: decreasing cosine, decreasing matched peaks, then ID.
        directed=pd.concat([
            ed.rename(columns={'feature_a':'node','feature_b':'neighbor'}),
            ed.rename(columns={'feature_b':'node','feature_a':'neighbor'})
        ],ignore_index=True)
        directed=directed.sort_values(['node','cosine','matched_peaks','neighbor'],
            ascending=[True,False,False,True])
        selected=directed.groupby('node',sort=False).head(mutual_n)
        # Require the directed selection in BOTH orientations.
        permitted=set(zip(selected.node.astype(str),selected.neighbor.astype(str)))
        ed=edges.loc[[((str(a),str(b)) in permitted and (str(b),str(a)) in permitted)
                      for a,b in zip(edges.feature_a,edges.feature_b)]].copy()
    for r in ed.itertuples(index=False):
        g.add_edge(str(r.feature_a),str(r.feature_b),cosine=float(r.cosine),
                   matched_peaks=int(r.matched_peaks))
    return g,ed

raw_net_all_ids=set(raw_net_records)
raw_net_threshold_graph, raw_net_threshold_edges=_make_raw_graph(
    raw_net_candidate_edges,raw_net_all_ids,mutual_n=None)
raw_net_mutual_graph, raw_net_mutual_edges=_make_raw_graph(
    raw_net_candidate_edges,raw_net_all_ids,mutual_n=RAW_NET_MUTUAL_TOP_N)

raw_net_networks={
    f'Mutual top-{RAW_NET_MUTUAL_TOP_N}':raw_net_mutual_graph,
    'Unpruned threshold':raw_net_threshold_graph,
}

raw_net_status_rows=[]
for label,g in raw_net_networks.items():
    hops=nx.single_source_shortest_path_length(g,raw_net_parent_id)
    for fid in sorted(raw_net_all_ids-set([raw_net_parent_id])):
        if raw_net_spectra[fid] is None:
            category='Insufficient peaks'
        elif fid not in hops:
            category='Disconnected'
        else:
            d=hops[fid]
            category='D1' if d == 1 else 'D2' if d == 2 else 'D3' if d == 3 else 'D4+'
        raw_net_status_rows.append({'graph':label,'feature_id':fid,
            'hop_distance':hops.get(fid,np.nan), 'raw_network_status':category,
            'raw_node_degree':g.degree(fid)})
raw_net_status_table=pd.DataFrame(raw_net_status_rows)
raw_net_status_table.to_csv(RAW_NET_DIR/'raw_network_feature_hop_distances.csv',index=False)
raw_net_threshold_edges.to_csv(RAW_NET_DIR/'raw_network_edges_unpruned.csv',index=False)
raw_net_mutual_edges.to_csv(RAW_NET_DIR/'raw_network_edges_mutual_top10.csv',index=False)

print('Network summaries:')
for label,g in raw_net_networks.items():
    reachable=len(nx.node_connected_component(g,raw_net_parent_id))
    print(f'{label}: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges; '
          f'{reachable} nodes in cilnidipine component')
print('Categories (mutual graph):')
display(raw_net_status_table.loc[raw_net_status_table.graph.eq(f'Mutual top-{RAW_NET_MUTUAL_TOP_N}'),
    'raw_network_status'].value_counts().to_frame('candidate_count'))


In [ ]:
# Use the *new, non-consensus raw modified-cosine network* for all case-study
# labels and novelty-based candidate selection. The graph is computed above.
# We use the mutual top-10 graph; the unpruned graph remains a sensitivity check.
case_network_label = f'Mutual top-{RAW_NET_MUTUAL_TOP_N}'
case_raw_status = (raw_net_status_table
    .loc[raw_net_status_table.graph.eq(case_network_label),
         ['feature_id', 'raw_network_status', 'hop_distance']]
    .drop_duplicates('feature_id'))
if case_raw_status.feature_id.duplicated().any():
    raise ValueError('Duplicate raw-network status for a feature')
pool_df = pool_df.drop(columns=['raw_network_status', 'raw_network_hops'], errors='ignore')
pool_df = pool_df.merge(
    case_raw_status.rename(columns={'hop_distance':'raw_network_hops'}),
    on='feature_id', how='left', validate='one_to_one')
if pool_df.raw_network_status.isna().any():
    raise RuntimeError('Missing raw-network category for pooled candidates')
# The original GNPS fields are retained for separate historical-network panels,
# but the case figures and their novelty criteria use these raw-network fields.
pool_df['novel_relative_to_D1'] = ~pool_df.raw_network_status.eq('D1')
pool_df['ranked_candidate_tier'] = np.select([
    pool_df.drug_specific_triage & pool_df.novel_relative_to_D1 &
        pool_df.plausible_mass_window_100Da & pool_df.fragment_matches_0p02Da.ge(3),
    pool_df.bio_support_proxy & pool_df.novel_relative_to_D1 & pool_df.plausible_mass_window_100Da,
    pool_df.bio_support_proxy & pool_df.novel_relative_to_D1],
    ['drug-associated + fragment support (exploratory)',
     'microbial + mass (exploratory)',
     'microbial + network-novel (exploratory)'], default='lower evidence')
shortlist=pool_df.nsmallest(min(CASE_TOP_N, len(pool_df)), 'raw_simba_rank').copy()
shortlist['tier_order']=shortlist.ranked_candidate_tier.map(priority)
shortlist=shortlist.sort_values(['tier_order', 'raw_simba_rank', 'abs_delta_mz'])
case_candidates=shortlist.head(CASE_COUNT).copy()
case_candidates['expert_annotation']='NOT REVIEWED — inspect adduct/isotope/structures manually'
case_candidates['confirmed_transformation']=False
case_export_columns = [
    'feature_id','raw_simba_rank','raw_cosine_rank','raw_network_status',
    'raw_network_hops','mz','delta_precursor_mz','raw_integer_mces_dist',
    'raw_sed_dist_class','raw_mces_dist','raw_cosine_median',
    'fragment_matches_0p02Da','bio_support_proxy','drug_specific_triage',
    'transformation_hint_unverified','expert_annotation','confirmed_transformation']
case_candidates[case_export_columns].to_csv(
    FIG_DIR/'two_candidate_raw_network_manual_review.csv', index=False)
print('Case studies now use:', case_network_label)
display(case_candidates[case_export_columns])


## 13A. Case-study abundance trajectories, raw mirror spectra, and mass/structure hypotheses

The time series uses MZmine peak **areas** (MS1), including Bac and noBac at T0–T8. The annotated mirror plot shows the **individual unmerged raw MS/MS scans** from the same injection, with fragment matches within ±0.02 Da. Green peaks denote m/z matches, not proven shared substructures. The evidence panel shows the predicted MCES and SED outputs but intentionally avoids plotting molecular structures without verified chemical identities.

Case-study network labels refer to the **raw modified-cosine network** (mutual top-10 graph), not the original GNPS network.


In [ ]:
def _plot_one_case(fid):
    fid=str(fid)
    row=pool_df.set_index('feature_id').loc[fid]
    fname,rec=chosen_records[fid]
    query=raw_query_by_file[fname]
    fig,axs=plt.subplots(1,3,figsize=(16.2,4.6),gridspec_kw={'width_ratios':[1.15,1.45,.85]})
    ax=axs[0]
    for group,color,label in [(MICROBIAL_GROUP,'#236B9A','Cilnidipine + bacteria'),
                              (ABIOTIC_GROUP,'#D77C28','Cilnidipine, no bacteria')]:
        times=sorted(sample_index.loc[sample_index.group.eq(group),'time_h'].unique())
        means=[];lows=[];highs=[]
        for t in times:
            cols=sample_index.loc[sample_index.group.eq(group)&sample_index.time_h.eq(t),'normalized_name'].tolist()
            vals=intensity.loc[fid,cols].to_numpy(dtype=float)
            z=np.log10(np.maximum(vals,0)+1)
            means.append(np.nanmedian(z));lows.append(np.nanmin(z));highs.append(np.nanmax(z))
        ax.plot(times,means,marker='o',lw=1.5,color=color,label=label)
        #ax.fill_between(times,lows,highs,alpha=.12,color=color)
    ax.set(xlabel='Experimental timepoint (T0–T8)',ylabel='log10(peak area + 1)',
           title='(a) MS1 abundance trajectories')
    ax.legend(fontsize=8,frameon=False)
    ax=axs[1]
    p=matched_fragments(query['mz'],rec['mz'],tol=FRAGMENT_TOL_DA)
    qmatched=np.array([a for a,b in p]);cmatched=np.array([b for a,b in p])
    for masses,intensities,sign,matched,basecol in [
        (query['mz'],query['intensity'],1,qmatched,'#8A8F95'),
        (rec['mz'],rec['intensity'],-1,cmatched,'#8A8F95')]:
        normalized=intensities/max(float(np.max(intensities)),1)
        flags=np.array([np.any(np.abs(m-matched)<=1e-6) for m in masses])
        ax.vlines(masses[~flags],0,sign*normalized[~flags],color=basecol,lw=.8,alpha=.62)
        ax.vlines(masses[flags],0,sign*normalized[flags],color='#168A68',lw=1.1)
    ax.axhline(0,color='black',lw=.6)
    ax.set(xlabel='Fragment m/z',ylabel='Relative intensity (query ↑, candidate ↓)',
           title=f'(b) Raw MS/MS mirror ({len(p)} matched peaks)')
    ax.text(.03,.95,'Green = matched m/z (±0.02 Da)',transform=ax.transAxes,
            va='top',fontsize=8,color='#168A68')
    ax=axs[2];ax.axis('off')
    txt=(f'Feature {fid}\n'
         f'Precursor m/z: {row.mz:.4f}\n'
         f'Δ precursor m/z: {row.delta_precursor_mz:+.4f} Da\n'
         f'SIMBA / cosine rank: {int(row.raw_simba_rank)} / {int(row.raw_cosine_rank)}\n'
         f'Modified cosine (pooled): {float(row.raw_cosine_median):.3f}\n'
         f'MCES predicted distance: {row.raw_mces_dist:.2f}\n'
         f'SED class (transformed): {int(row.raw_sed_dist_class)}\n'
         f'Raw cosine network: {row.raw_network_status}\n'
         f'Biological proxy: {bool(row.bio_support_proxy)}\n'
         f'DMSO triage: {bool(row.drug_specific_triage)}\n\n'
         f'Mass-shift hint ONLY:\n{row.transformation_hint_unverified}\n\n'
         'Requires manual chemical review')
    ax.text(0,1,txt,ha='left',va='top',fontsize=9,linespacing=1.5)
    fig.suptitle(f'Putative raw-MS/MS example: feature {fid} (no structural confirmation)',fontsize=12)
    fig.tight_layout();save_figure(fig,f'FigC_case_feature_{fid}');plt.show()

if case_candidates.empty:
    print('No case candidates available')
else:
    for fid in case_candidates.feature_id:
        if str(fid) in chosen_records:_plot_one_case(fid)
        else:print('No saved raw scan for candidate:',fid)


## 13B. Precursor-mass plausibility and SIMBA-vs-cosine ranking disagreements

This panel helps select chemically credible candidates without equating m/z proximity with a verified transformation. The dotted ±100 Da region is a **visualization heuristic**, not a retrieval restriction. Colors describe MS1 evidence and markers describe original-network novelty. Precursor m/z shifts must not be reported as neutral molecular transformations unless the charge states/adducts are verified.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12.8,4.7))
sub=pool_df.nsmallest(min(100,len(pool_df)),'raw_simba_rank').copy()
ax=axs[0]
for status,marker in [('D1','o'),('distal','s'),('disconnected','^'),('not_in_graph','X')]:
    for supported,color in [(True,'#168A68'),(False,'#AEB2B8')]:
        d=sub.loc[(sub.network_status==status)&(sub.drug_specific_triage==supported)]
        if len(d):ax.scatter(d.delta_precursor_mz,d.raw_mces_dist,marker=marker,s=35,
                            color=color,alpha=.8,label=f'{status}: '+('drug triage' if supported else 'other'))
ax.axvspan(-100,100,color='#236B9A',alpha=.055)
ax.axvline(0,color='black',alpha=.4,lw=1)
ax.set(xlabel='Candidate − parent precursor m/z (Da)',
       ylabel='SIMBA predicted MCES distance',
       title='(a) SIMBA top 100: precursor shifts vs predicted distance')
ax.legend(fontsize=6,ncol=2,frameon=False,loc='best')
ax=axs[1]
ax.scatter(pool_df.raw_simba_rank,pool_df.raw_cosine_rank,
           s=10,color='#A4ADB5',alpha=.27,label='Other')
positive=pool_df.loc[pool_df.drug_specific_triage]
ax.scatter(positive.raw_simba_rank,positive.raw_cosine_rank,s=22,
           color='#168A68',alpha=.65,label='Drug-specific triage')
for r in case_candidates.itertuples(index=False):
    ax.annotate(str(r.feature_id),(r.raw_simba_rank,r.raw_cosine_rank),
                xytext=(4,4),textcoords='offset points',fontsize=8)
ax.plot([1,len(pool_df)],[1,len(pool_df)],ls='--',lw=.8,color='gray')
ax.set(xlabel='SIMBA rank (smaller is better)',ylabel='Modified cosine rank',
       xscale='log',yscale='log',title='(b) Rank disagreements on same candidates')
ax.legend(frameon=False,fontsize=8)
fig.tight_layout();save_figure(fig,'FigD_mass_shift_and_rank_disagreement');plt.show()


## 14. Compact multi-panel summary for the manuscript

Panel A: Top-10 biological support; Panel B: network status; Panel C: DMSO-control-supported triage. These are explicitly exploratory feature **prioritization metrics**, not confirmed metabolite identification rates. The panel uses the same pooled T8 Bac raw-MS/MS candidate universe for every included method.

In [ ]:
names=list(method_ranks)
fig,axes=plt.subplots(1,3,figsize=(15,4.4))
ax=axes[0]
counts=[int(pool_df.nsmallest(10,col).bio_support_proxy.sum()) for col in method_ranks.values()]
ax.bar(names,counts,color=[COLORS.get(x,'#888888') for x in names])
for i,v in enumerate(counts):ax.text(i,v+.2,f'{v}/10',ha='center')
ax.set(title='(a) Top-10 microbial enrichment',ylabel='Features passing proxy',ylim=(0,11))
ax.tick_params(axis='x',rotation=23)
ax=axes[1]
net=network_top.loc[network_top.k.eq(10)]
left=np.zeros(len(names))
for status,color in net_pal.items():
    vals=np.array([net.loc[(net.method.eq(x))&(net.network_status.eq(status)),'n'].sum() for x in names])
    ax.bar(names,vals,bottom=left,color=color,label=status)
    left+=vals
ax.set(title='(b) Original GNPS network status',ylabel='Top-10 features',ylim=(0,11))
ax.tick_params(axis='x',rotation=23)
ax.legend(fontsize=7,frameon=False,loc='upper right')
ax=axes[2]
vals=[int(pool_df.nsmallest(10,col).drug_specific_triage.sum()) for col in method_ranks.values()]
ax.bar(names,vals,color=[COLORS.get(x,'#888888') for x in names])
for i,v in enumerate(vals):ax.text(i,v+.2,f'{v}/10',ha='center')
ax.set(title='(c) Additional DMSO+Bac control',ylabel='Features passing stricter triage',ylim=(0,11))
ax.tick_params(axis='x',rotation=23)
fig.suptitle('Cilnidipine T8: raw-query and raw-candidate spectral retrieval (exploratory)',fontsize=12)
fig.tight_layout();save_figure(fig,'FigMain_raw_top10_network_biology');plt.show()

# Versioning/audit of exact requirements and evidence.
metadata={'parent_feature_id':PARENT_FEATURE_ID,'primary_pool':PRIMARY_POOL,
          'n_candidate_features':len(pool_df),'ranking_columns':method_ranks,
          'raw_query':True,'raw_candidates':True,'network_edge_file':str(EDGE_FILE),
          'biological_label':'T8 Bac vs abiotic and differential T0->T8 fold proxy',
          'additional_control':'T8 Cilni Bac vs T8 DMSO Bac; detection >=2 Bac replicates',
          'important_limitation':'no confirmed metabolite identities; no direct adduct verification; missing baseline raw scores omitted'}
(FIG_DIR/'manuscript_analysis_settings.json').write_text(json.dumps(metadata,indent=2,default=str))
print('Manuscript figure files:',len(list(FIG_DIR.glob('*.pdf'))),'PDF figures in',FIG_DIR)
print('All methods represented:',names)


## 15. Article figure: biological enrichment at Top 5, Top 10 and Top 20

This grouped bar chart compares **SIMBA and modified cosine on the same pooled Bac raw-MS/MS candidate set**, using the existing `bio_support_proxy` (microbial-versus-abiotic abundance criterion). A second companion panel evaluates the additional **DMSO+Bac triage**. The plots report counts and percentages, not structurally verified metabolite identifications. No values are hard-coded: all results are recomputed from `pool_df` and its ranking columns.


In [ ]:
# Manuscript figure 1: Top-5/10/20 biological enrichment, pooled T8 + bacteria.
from matplotlib.ticker import MultipleLocator

ARTICLE_METHODS = [('SIMBA', 'raw_simba_rank'), ('Modified cosine', 'raw_cosine_rank')]
ARTICLE_K = (5, 10, 20)
ARTICLE_COLORS = {'SIMBA':'#236B9A', 'Modified cosine':'#D77C28'}

required = {'feature_id', 'bio_support_proxy', 'drug_specific_triage',
            'raw_simba_rank', 'raw_cosine_rank'}
missing = required - set(pool_df.columns)
if missing:
    raise RuntimeError(f'Run the earlier network/biology analysis cells first; missing: {sorted(missing)}')
if len(pool_df) < max(ARTICLE_K):
    raise RuntimeError(f'Not enough eligible candidates ({len(pool_df)}) for Top {max(ARTICLE_K)}')

article_rows=[]
for k in ARTICLE_K:
    for method, rank_col in ARTICLE_METHODS:
        selected=pool_df.nsmallest(k,rank_col)
        article_rows.append(dict(method=method, top_k=k, n_candidates=k,
                                 microbial_enriched=int(selected.bio_support_proxy.fillna(False).sum()),
                                 drug_specific_triage=int(selected.drug_specific_triage.fillna(False).sum())))
article_enrichment=pd.DataFrame(article_rows)
article_enrichment.to_csv(FIG_DIR/'Fig15_top5_10_20_biological_enrichment.csv',index=False)
display(article_enrichment)

def _article_enrichment_bars(metric, y_label, stem, caption):
    fig,ax=plt.subplots(figsize=(8.6,5.0))
    xpos=np.arange(len(ARTICLE_K),dtype=float)
    width=0.34
    for i,(method,_) in enumerate(ARTICLE_METHODS):
        subset=article_enrichment.set_index(['top_k','method'])
        counts=np.array([int(subset.loc[(k,method),metric]) for k in ARTICLE_K])
        percents=counts/np.array(ARTICLE_K)*100
        bars=ax.bar(xpos+(i-0.5)*width, percents, width, color=ARTICLE_COLORS[method],
                    label=method,edgecolor='white',linewidth=.6)
        for bar,num,k,percentage in zip(bars,counts,ARTICLE_K,percents):
            ax.annotate(f'{num}/{k}',(bar.get_x()+bar.get_width()/2,bar.get_height()),
                        xytext=(0,5),textcoords='offset points',ha='center',va='bottom',
                        fontsize=10,fontweight='bold')
    ax.set(xticks=xpos,xticklabels=[f'Top {k}' for k in ARTICLE_K],
           ylabel=y_label,ylim=(0,115))
    ax.yaxis.set_major_locator(MultipleLocator(20))
    ax.legend(frameon=False,loc='upper right',ncol=2)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',alpha=.17,zorder=0)
    ax.set_axisbelow(True)
    fig.text(.5,.01,caption,ha='center',fontsize=8,color='#444444')
    fig.tight_layout(rect=[0,.035,1,1])
    save_figure(fig,stem)
    plt.show()

_article_enrichment_bars('microbial_enriched',
    'Retrieved features satisfying microbial enrichment (%)',
    'Fig15_raw_top5_10_20_microbial_enrichment',
    'T8 pooled Bac; raw query and candidates; abundance proxy, not confirmed metabolites.')

_article_enrichment_bars('drug_specific_triage',
    'Retrieved features passing DMSO-aware triage (%)',
    'Fig15b_raw_top5_10_20_dmso_triage',
    'T8 pooled Bac; adds T8 DMSO+Bac comparison and replicate detection criterion.')


## 16. Article figure: two SIMBA-prioritized candidate examples

Two rows show features **11184 and 8742** (if present, otherwise the top two automatically shortlisted candidates). Each row displays (A) Bac and noBac T0–T8 median MS1 peak-area trajectories, (B) the raw MS/MS mirror spectrum, and (C) SIMBA predictions, precursor mass differences and **distance in the reconstructed raw-MS/MS modified-cosine network** (mutual top-10 graph). No consensus-spectrum GNPS network annotation is used in this case-study figure.

The evidence panels report the **pooled modified-cosine score** used for feature ranking (higher = more similar).

In [ ]:
# Manuscript figure 2: combined, publication-oriented three-case comparison.
ARTICLE_CASE_IDS = ['11184', '8742']  # previously highlighted candidates
available={str(fid) for fid in pool_df.feature_id}
requested=[fid for fid in ARTICLE_CASE_IDS if fid in available and fid in chosen_records]
for fid in case_candidates.feature_id.astype(str):
    if len(requested)>=2: break
    if fid in available and fid in chosen_records and fid not in requested:
        requested.append(fid)
if len(requested)<2:
    raise RuntimeError(f'Only {len(requested)} candidate raw scans are available for the two-case figure')
ARTICLE_CASE_IDS=requested[:2]
print('Two displayed candidates:',ARTICLE_CASE_IDS)

case_lookup=pool_df.set_index('feature_id')
fig, axes=plt.subplots(2,3,figsize=(16.0,9.3),
    gridspec_kw={'width_ratios':[1.1,1.4,.98]},constrained_layout=True)
case_export=[]
for ridx,fid in enumerate(ARTICLE_CASE_IDS):
    row=case_lookup.loc[fid]
    fname,rec=chosen_records[fid]
    query=raw_query_by_file[fname]
    assert len(query['mz']) and len(rec['mz'])
    # (Left) Quantification from original feature table: median and replicate range.
    ax=axes[ridx,0]
    for group,color,label in [(MICROBIAL_GROUP,'#236B9A','Cilnidipine + Bac'),
                              (ABIOTIC_GROUP,'#D77C28','Cilnidipine + noBac')]:
        times=sorted(sample_index.loc[sample_index.group.eq(group),'time_h'].unique())
        time_vals=[];center=[];lower=[];upper=[]
        for t in times:
            cols=sample_index.loc[sample_index.group.eq(group)&sample_index.time_h.eq(t),
                                  'normalized_name'].tolist()
            if not cols: continue
            vals=pd.to_numeric(intensity.loc[fid,cols],errors='coerce').to_numpy(float)
            vals=np.log10(np.maximum(vals,0)+1)
            if not np.isfinite(vals).any():continue
            time_vals.append(t)
            center.append(np.nanmedian(vals));lower.append(np.nanmin(vals));upper.append(np.nanmax(vals))
        ax.plot(time_vals,center,marker='o',ms=3.5,lw=1.5,color=color,label=label)
        #ax.fill_between(time_vals,lower,upper,color=color,alpha=.14)
    ax.set(xlabel='Timepoint (T0–T8)',ylabel='log10(peak area + 1)',
           title=f'Feature {fid} | abundance')
    if ridx==0:ax.legend(loc='best',fontsize=8,frameon=False)
    ax.spines[['top','right']].set_visible(False)
    # (Middle) Mirrors of actual selected MS2 query/candidate scans from same injection.
    ax=axes[ridx,1]
    pairs=matched_fragments(query['mz'],rec['mz'],tol=FRAGMENT_TOL_DA)
    qmatched=np.array([x for x,_ in pairs],dtype=float)
    cm= np.array([y for _,y in pairs],dtype=float)
    for masses,intensities,sgn,matched in [
        (query['mz'],query['intensity'],1,qmatched),
        (rec['mz'],rec['intensity'],-1,cm)]:
        mz=np.asarray(masses,dtype=float)
        vals=np.asarray(intensities,dtype=float)
        scaled=vals/max(float(np.nanmax(vals)),1.)
        mask=np.zeros(len(mz),dtype=bool)
        for mt in matched:mask |= (np.abs(mz-mt)<=1e-6)
        ax.vlines(mz[~mask],0,sgn*scaled[~mask],color='#8A8F95',lw=.75,alpha=.65)
        ax.vlines(mz[mask],0,sgn*scaled[mask],color='#168A68',lw=1.05)
    ax.axhline(0,color='#404040',lw=.65)
    ax.set(xlabel='Fragment m/z',ylabel='Relative intensity (parent ↑ / hit ↓)',
           title=f'Raw MS/MS | {len(pairs)} shared fragments')
    ax.set_ylim(-1.12,1.12)
    ax.spines[['top','right']].set_visible(False)
    # (Right) Explicit evidence without assigning an unverified chemical structure.
    ax=axes[ridx,2]
    ax.axis('off')
    summary=(
      f'Feature {fid}\n'
      f'SIMBA rank: {int(row.raw_simba_rank)}\n'
      f'Mod. cosine rank: {int(row.raw_cosine_rank)}\n'
      f'Mod. cosine score: {float(row.raw_cosine_median):.3f}\n'
      f'Precursor m/z: {float(row.mz):.4f}\n'
      f'Δ precursor: {float(row.delta_precursor_mz):+.4f} Da\n'
      f'SIMBA MCES distance: {float(row.raw_mces_dist):.2f}\n'
      f'SIMBA SED distance: {int(row.raw_sed_dist_class)}\n'
      f'Raw cosine network: {row.raw_network_status}\n'
      f'Bac/abiotic enriched: {bool(row.bio_support_proxy)}\n'
      f'DMSO triage: {bool(row.drug_specific_triage)}\n\n'
    )
    ax.text(.02,.98,summary,ha='left',va='top',fontsize=9.3,linespacing=1.5,
            transform=ax.transAxes)
    case_export.append({'feature_id':fid,'simba_rank':int(row.raw_simba_rank),
        'cosine_rank':int(row.raw_cosine_rank),
        'cosine_score_pooled':float(row.raw_cosine_median),
        'raw_network_status':row.raw_network_status,
        'precursor_mz':row.mz,'delta_precursor_mz':row.delta_precursor_mz,
        'shared_fragment_count':len(pairs),'raw_scan_filename':fname,
        'drug_specific_triage':bool(row.drug_specific_triage),
        'transformation_hint_unverified':row.transformation_hint_unverified})
fig.suptitle('Cilnidipine: two SIMBA-prioritized features (raw MS/MS; tentative assignments)',
             fontsize=15,fontweight='bold')
save_figure(fig,'Fig16_two_raw_msms_candidate_examples')
plt.show()
case_table=pd.DataFrame(case_export)
case_table.to_csv(FIG_DIR/'Fig16_two_candidate_metadata.csv',index=False)
display(case_table)
print('Figures exported as PDF, SVG, PNG to:', FIG_DIR)


## How to describe these results appropriately

**Network novelty:** Report the count of raw SIMBA top-10 and top-20 features outside the original GNPS2 parent D1 neighborhood, distinguishing distal, disconnected and not-in-network features. These are *candidates absent from a cosine network neighborhood*, not confirmed structural analogs.

**Biological support:** Show T0–T8 replicate peak-area trajectories and whether candidates exceed both abiotic cilnidipine and bacterial DMSO controls. Peak-area shifts are associations; require replication, adduct checks, blank/QC filtering and ideally formal hypothesis testing.

**Case studies:** Use the three exported case-study figures and the `three_candidate_manual_review_template.csv` for independent manual MS/MS and molecular annotation. Do not label the transformation hypotheses as confirmed unless supported by precursor formula/adduct, neutral-loss/fragment assignments, and orthogonal identification.

**Learned baselines:** Include DreaMS, MS2DeepScore and Spec2Vec only after scoring the *same individual raw spectra* or documenting a strictly comparable raw-spectrum workflow. The panels explicitly omit any missing baseline rather than falsely claiming their evaluation.


---

## 17. Two SIMBA-prioritized examples selected from the final ten-example set

**The two examples in Fig. 17 are always a subset of the exact ten features exported and plotted in Sections 19–20.** The final ten are defined in the next code cell, by ascending SIMBA rank, among candidates with **0.30 ≤ pooled modified-cosine score < 0.70**, cosine rank worse than SIMBA rank, and an available raw MS/MS spectrum. Cosine scores below 0.30 are excluded. Neither the ten-example nor two-example selection rewards the lowest cosine score.

Within those ten candidates, choose two examples by prioritizing previously defined independent evidence (DMSO-aware enrichment, mass/fragment clues and non-D1 network position) and then SIMBA rank. Unlike an additional hard filter, these preferences do not prevent selecting exactly two of the ten. The audit CSV explicitly reports which stronger evidence criteria each candidate passes; **the selected features are not necessarily confirmed drug metabolites**.

The 0.30–0.70 threshold refers to **pooled modified-cosine retrieval scores**. Individual-scan cosine scores (shown separately in case panels) can differ, and network connectivity uses different preprocessing, tolerances and reciprocal pruning. Mass shifts are precursor m/z differences, not verified neutral-mass transformations.


In [ ]:
# Screen all candidates reproducibly; never pick examples based on desired results alone.
LOW_COSINE_MIN = 0.00  # inclusive: pooled modified cosine must be >= 0.30
LOW_COSINE_MAX = 0.70  # exclusive: pooled modified cosine must be < 0.70
GOOD_SIMBA_MAX_RANK = 150
MAX_SIMBA_MCES_DISTANCE = 12.0  # exploratory structural-closeness threshold; verify calibration
MIN_RANK_DISAGREEMENT = 20
MAX_MASS_DISTANCE_DA = 50.0
MATCH_TOL_DA = 0.025
EXCLUDED_OLD_EXAMPLES = {'11184','8742','11005'}

# Additional possible simple shifts (the original MASS_HINTS are retained).
EXTRA_MASS_HINTS = {
    'NH2 addition (nominal)': 16.01872,
    'NH3 loss': -17.02655,
    'NH3 addition': 17.02655,
    'CO2 loss (decarboxylation)': -43.98983,
    'CO2 addition': 43.98983,
    'CH3OH loss': -32.02621,
    'CH2O addition': 30.01056,
    'CH2O loss': -30.01056,
    'C2H2 addition': 26.01565,
    'C2H4 addition': 28.03130,
    'C2H4 loss': -28.03130,
    'C2H2O addition': 42.01056,
    'C2H2O loss': -42.01056,
}
ALL_MASS_HINTS = {**MASS_HINTS, **EXTRA_MASS_HINTS}

def _possible_shift(delta):
    if not np.isfinite(delta):
        return ('none', np.nan)
    ranked=sorted((abs(delta-val),label) for label,val in ALL_MASS_HINTS.items())
    if ranked and ranked[0][0] <= MATCH_TOL_DA:
        return (ranked[0][1],ranked[0][0])
    return ('no common single-step shift within tolerance', ranked[0][0] if ranked else np.nan)

def _neutral_loss_matches(query,hit,tol=0.02):
    # A conserved neutral loss is supportive fragmentation evidence but not structure proof.
    q=np.asarray(query['precursor_mz']-np.asarray(query['mz'],float),float)
    h=np.asarray(hit['precursor_mz']-np.asarray(hit['mz'],float),float)
    q=q[np.isfinite(q) & (q>0)]
    h=h[np.isfinite(h) & (h>0)]
    return len(matched_fragments(q,h,tol=tol))

additional_screen=pool_df.copy()
additional_screen['feature_id']=additional_screen.feature_id.astype(str)
additional_screen['cosine_rank_minus_simba_rank']=(additional_screen.raw_cosine_rank-additional_screen.raw_simba_rank)
additional_screen['shift_label']=additional_screen.delta_precursor_mz.apply(lambda x:_possible_shift(x)[0])
additional_screen['shift_mass_error_da']=additional_screen.delta_precursor_mz.apply(lambda x:_possible_shift(x)[1])
additional_screen['named_mass_shift']=~additional_screen.shift_label.isin(['none','no common single-step shift within tolerance'])
neutral=[]
for fid in additional_screen.feature_id:
    if fid in chosen_records:
        fname,scan=chosen_records[fid]
        neutral.append(_neutral_loss_matches(raw_query_by_file[fname],scan))
    else:
        neutral.append(np.nan)
additional_screen['shared_neutral_losses_0p02Da']=neutral
additional_screen['low_cosine']=additional_screen.raw_cosine_median.ge(LOW_COSINE_MIN) & additional_screen.raw_cosine_median.lt(LOW_COSINE_MAX)
additional_screen['simba_early']=(additional_screen.raw_simba_rank.le(GOOD_SIMBA_MAX_RANK) &
    additional_screen.raw_mces_dist.le(MAX_SIMBA_MCES_DISTANCE))
additional_screen['large_rank_disagreement']=additional_screen.cosine_rank_minus_simba_rank.ge(MIN_RANK_DISAGREEMENT)
additional_screen['bio_control_supported']=additional_screen.drug_specific_triage.fillna(False).astype(bool)
additional_screen['fragment_clue']=(additional_screen.fragment_matches_0p02Da.fillna(0).ge(1) |
                                     additional_screen.shared_neutral_losses_0p02Da.fillna(0).ge(1))
additional_screen['mass_clue']=(additional_screen.named_mass_shift |
     (additional_screen.abs_delta_mz.le(MAX_MASS_DISTANCE_DA) & additional_screen.fragment_clue))
additional_screen['not_d1']=~additional_screen.raw_network_status.eq('D1')
additional_screen['has_raw_scan']=additional_screen.feature_id.isin(chosen_records)
additional_screen['excluded_previous_example']=additional_screen.feature_id.isin(EXCLUDED_OLD_EXAMPLES)
filter_columns=['low_cosine','simba_early','large_rank_disagreement','bio_control_supported',
                'mass_clue','not_d1','has_raw_scan']
additional_screen['passes_all_criteria']=additional_screen[filter_columns].all(axis=1) & ~additional_screen.excluded_previous_example
# Define the final ten exported candidates HERE, before the case-study figure.
# Sections 19 and 20 reuse this *exact* ordered selection. All cosine threshold
# checks refer to the POOLED modified-cosine score, not a single plotted scan.
MGF_N_CANDIDATES = 10
MGF_MIN_COSINE = LOW_COSINE_MIN
MGF_MAX_COSINE = LOW_COSINE_MAX
mgf_ranked = (additional_screen.loc[
    additional_screen.raw_cosine_median.ge(MGF_MIN_COSINE)
    & additional_screen.raw_cosine_median.lt(MGF_MAX_COSINE)
    & additional_screen.raw_cosine_rank.gt(additional_screen.raw_simba_rank)
    & additional_screen.has_raw_scan
].sort_values(['raw_simba_rank', 'raw_cosine_rank', 'feature_id'])
 .head(MGF_N_CANDIDATES).copy())
if len(mgf_ranked) != MGF_N_CANDIDATES:
    raise RuntimeError(
        f'Only {len(mgf_ranked)} candidates satisfy 0.30 <= pooled cosine < 0.70, '
        f'cosine rank > SIMBA rank, and raw-scan availability; need {MGF_N_CANDIDATES}. '
        'Inspect the candidate pool rather than relaxing a threshold silently.'
    )
if not mgf_ranked.feature_id.is_unique:
    raise RuntimeError('Repeated feature ID in the ten-example export.')

# The two illustrative features MUST be a subset of the ten exported features.
# Prefer independent controls/fragment-mass evidence and early SIMBA ranks,
# while NOT making those extra evidence properties mandatory: this guarantees
# two examples whenever the final ten-export cohort can be produced.
additional_screen['evidence_tier'] = np.select([
    additional_screen.passes_all_criteria,
    additional_screen.bio_control_supported & additional_screen.mass_clue & additional_screen.not_d1,
    additional_screen.bio_control_supported & additional_screen.not_d1,
    additional_screen.bio_control_supported], [0, 1, 2, 3], default=4)
additional_shortlist=(additional_screen.loc[
    additional_screen.feature_id.isin(mgf_ranked.feature_id)
].sort_values(['evidence_tier', 'raw_simba_rank', 'feature_id']).copy())
ADDITIONAL_CASE_IDS=additional_shortlist.feature_id.head(2).tolist()
assert len(ADDITIONAL_CASE_IDS) == 2
assert set(ADDITIONAL_CASE_IDS).issubset(set(mgf_ranked.feature_id))
assert all(LOW_COSINE_MIN <= float(v) < LOW_COSINE_MAX for v in
           additional_shortlist.head(2).raw_cosine_median)
print('Canonical 10 exported features:', mgf_ranked.feature_id.tolist())
print('Two figure examples (subset of those 10):', ADDITIONAL_CASE_IDS)
print('Selection prioritizes full evidence criteria; extra biological screens '
      'are not guaranteed for every selected feature.')
cols=['feature_id','raw_simba_rank','raw_cosine_rank','raw_cosine_median','raw_mces_dist',
      'raw_sed_dist_class','mz','delta_precursor_mz','shift_label','shift_mass_error_da',
      'fragment_matches_0p02Da','shared_neutral_losses_0p02Da',
      'raw_network_status','bio_support_proxy','drug_specific_triage',
      'log2_cilni_bac_vs_dmso_bac_T8','cosine_rank_minus_simba_rank',
      'passes_all_criteria','excluded_previous_example']
additional_screen[cols].sort_values('raw_simba_rank').to_csv(
    FIG_DIR/'Fig17_low_cosine_full_candidate_audit.csv',index=False)
additional_shortlist[cols].to_csv(FIG_DIR/'Fig17_low_cosine_shortlist.csv',index=False)
print(f'Screened {len(additional_screen)} features; {int(additional_screen.passes_all_criteria.sum())} meet ALL stringent criteria overall.')
print('Canonical ten: pooled cosine in [',LOW_COSINE_MIN,',',LOW_COSINE_MAX,') and cosine rank > SIMBA rank')
print('Two selected from those ten, prioritizing existing bio/mass/network evidence and SIMBA rank')
print('Selected figure examples from final ten:', ADDITIONAL_CASE_IDS)
display(additional_shortlist[cols].head(20))
# Evidence tier/criteria are exported, including when a selected feature is not fully supported.

## 17A. Article figure: two examples from the final ten, with 0.30 ≤ pooled modified cosine < 0.70

Each row shows **(A)** T0–T8 median MS1 peak areas for Cilnidipine+Bac and Cilnidipine+noBac, **(B)** an individual raw MS/MS mirror plot, and **(C)** SIMBA/cosine ranks, **pooled** modified-cosine score (used to enforce 0.30 ≤ score < 0.70), individual-scan score, precursor m/z, Δm/z and raw network connectivity. The two feature IDs must also appear in the ten mirror plots in Section 20. Independent biological and chemical evidence is shown explicitly and is not treated as a confirmed structural annotation.


In [ ]:
from matplotlib.ticker import MaxNLocator

def _plot_extra_two_cases(ids):
    if not ids:
        print('No cases meet the criteria; no illustrative figure generated. Full audit CSV saved.')
        return
    fig,axes=plt.subplots(len(ids),3,figsize=(17.5,5.0*len(ids)),squeeze=False,
        gridspec_kw={'width_ratios':[1.15,1.55,1.08]},constrained_layout=True)
    case_rows=[]
    lookup=additional_screen.set_index('feature_id')
    conditions=[('Cilnidipine + Bac','#236B9A','Cilni_Bac'),
                ('Cilnidipine + noBac','#D77C28','Cilni_noBac')]
    for i,fid in enumerate(ids):
        row=lookup.loc[fid]
        filename,hit=chosen_records[fid]
        query=raw_query_by_file[filename]
        ax=axes[i,0]
        quant=ft_by_id.loc[fid]
        for display_name,color,key in conditions:
            t_all=[];ys=[];lo=[];hi=[]
            for t in range(9):
                names=[f't{t}_{key.lower()}_rep{rep}' for rep in (1,2,3)]
                src_cols=[normalized_ft_cols[n] for n in names if n in normalized_ft_cols]
                if not src_cols:continue
                values=pd.to_numeric(quant[src_cols],errors='coerce').to_numpy(dtype=float)
                if not np.isfinite(values).any():continue
                transformed=np.log10(np.maximum(values,0)+1)
                t_all.append(t)
                ys.append(float(np.nanmedian(transformed)))
                lo.append(float(np.nanmin(transformed)))
                hi.append(float(np.nanmax(transformed)))
            if t_all:
                ax.plot(t_all,ys,'-o',ms=3.3,lw=1.55,color=color,label=display_name)
                #ax.fill_between(t_all,lo,hi,color=color,alpha=0.12)
        ax.set(xlabel='Experimental timepoint (T0–T8)',ylabel='log10(MS1 peak area + 1)',
               title=f'{chr(65+i*3)}. Feature {fid}: quantitative evidence')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True,nbins=9))
        if i==0: ax.legend(frameon=False,fontsize=8,loc='best')
        ax.spines[['right','top']].set_visible(False)

        ax=axes[i,1]
        matched=matched_fragments(query['mz'],hit['mz'],tol=FRAGMENT_TOL_DA)
        qcommon=np.array([p[0] for p in matched],float)
        hcommon=np.array([p[1] for p in matched],float)
        for spec,sign,common in [(query,+1,qcommon),(hit,-1,hcommon)]:
            mz=np.asarray(spec['mz'],float)
            values=np.asarray(spec['intensity'],float)
            normalized=values/max(1,float(np.nanmax(values)))
            hitmask=np.isin(mz,common)
            ax.vlines(mz[~hitmask],0,sign*normalized[~hitmask],color='#8F949B',lw=.65,alpha=.65)
            ax.vlines(mz[hitmask],0,sign*normalized[hitmask],color='#158366',lw=1.2)
        ax.axhline(0,color='#333333',lw=.7)
        ax.set(xlabel='Fragment m/z',ylabel='Relative intensity: query ↑ / candidate ↓',
               ylim=(-1.12,1.12),title=f'{chr(66+i*3)}. Raw mirror: {len(matched)} shared peaks (±{FRAGMENT_TOL_DA:g} Da)')
        ax.spines[['right','top']].set_visible(False)

        ax=axes[i,2]; ax.axis('off')
        rs=full_primary_scans.loc[(full_primary_scans.filename==filename) &
                                   (full_primary_scans.feature_id.astype(str)==fid)]
        indiv_cosine=float(rs.iloc[0]['raw_cosine']) if len(rs) else np.nan
        text=(f'Feature {fid}\n'
              f'SIMBA rank: {int(row.raw_simba_rank)}\n'
              f'Cosine rank: {int(row.raw_cosine_rank)}\n'
              f'Mod. cosine (pooled): {float(row.raw_cosine_median):.3f}\n'
              f'Mod. cosine (shown scan): {indiv_cosine:.3f}\n'
              f'SIMBA MCES distance: {row.raw_mces_dist:.2f}\n'
              f'SIMBA SED distance: {int(row.raw_sed_dist_class)}\n'
              f'Precursor m/z: {row.mz:.4f}\n'
              f'Δm/z: {row.delta_precursor_mz:+.4f} Da\n'
              #f'Mass shift hint: {row.shift_label}\n'
              #f'Neutral-loss matches: {int(row.shared_neutral_losses_0p02Da)}\n'
              f'Raw cosine network: {row.raw_network_status}\n'
              f'DMSO-aware enrichment: {bool(row.drug_specific_triage)}\n'
              f'All strict evidence criteria: {bool(row.passes_all_criteria)}\n'
              f'Included in final ten: {fid in set(mgf_ranked.feature_id)}\n\n')
        ax.text(.02,.99,text.replace('\\n','\n'),transform=ax.transAxes,ha='left',va='top',
                fontsize=9.1,linespacing=1.45)
        case_rows.append({'feature_id':fid,'simba_rank':int(row.raw_simba_rank),
            'cosine_rank':int(row.raw_cosine_rank),'cosine_pooled':float(row.raw_cosine_median),
            'cosine_shown_scan':indiv_cosine,'raw_mces_distance':float(row.raw_mces_dist),
            'delta_mz_da':float(row.delta_precursor_mz),
            'mass_shift_hint_unconfirmed':row.shift_label,
            'shared_fragments_0p02da':len(matched),
            'shared_neutral_losses_0p02da':int(row.shared_neutral_losses_0p02Da),
            'raw_network_status':row.raw_network_status,'dmso_triage':bool(row.drug_specific_triage),
            'raw_scan_file':filename,'raw_scan_id':hit['scan_id'],
            'chemical_identity_confirmed':False})
    fig.suptitle('Two of the final ten SIMBA candidates (0.30 ≤ pooled cosine < 0.70)',
                 fontsize=14,fontweight='bold')
    save_figure(fig,'Fig17_two_additional_low_cosine_simBALinks')
    plt.show()
    out=pd.DataFrame(case_rows)
    out.to_csv(FIG_DIR/'Fig17_two_additional_low_cosine_case_metadata.csv',index=False)
    display(out)
    return out

extra_cases=_plot_extra_two_cases(ADDITIONAL_CASE_IDS)
print('Figure PDF / SVG / PNG and transparent selection audit saved in:',FIG_DIR)

In [ ]:
# Context panel: show where the selected examples lie among ALL scored raw candidates.
fig,ax=plt.subplots(figsize=(7.6,5.3))
background=additional_screen.loc[np.isfinite(additional_screen.raw_cosine_median) &
                                 np.isfinite(additional_screen.raw_mces_dist)]
ax.scatter(background.raw_cosine_median,background.raw_mces_dist,s=12,
           color='#97A6B2',alpha=.28,edgecolors='none',label='All eligible raw candidates')
if ADDITIONAL_CASE_IDS:
    selected=additional_screen.set_index('feature_id').loc[ADDITIONAL_CASE_IDS]
    ax.scatter(selected.raw_cosine_median,selected.raw_mces_dist,s=95,
               color='#C65335',edgecolor='white',linewidth=1.2,zorder=5,label='Selected case examples')
    for fid,r in selected.iterrows():
        ax.annotate(str(fid),(r.raw_cosine_median,r.raw_mces_dist),
                    xytext=(6,5),textcoords='offset points',fontsize=9,fontweight='bold')
ax.axvline(LOW_COSINE_MIN,color='#C65335',linestyle='--',linewidth=1,
           label=f'Minimum pooled cosine (≥ {LOW_COSINE_MIN:g})')
ax.axvline(LOW_COSINE_MAX,color='#C65335',linestyle='--',linewidth=1,
           label=f'Cosine threshold (< {LOW_COSINE_MAX:g})')
ax.axhline(MAX_SIMBA_MCES_DISTANCE,color='#3C7B9D',linestyle=':',linewidth=1,
           label='SIMBA MCES distance screen')
ax.set(xlabel='Pooled modified-cosine score (higher = more similar)',
       ylabel='SIMBA predicted MCES distance (lower = closer)',
       title='SIMBA–cosine disagreement across the raw candidate pool')
ax.spines[['top','right']].set_visible(False)
ax.legend(frameon=False,fontsize=8)
fig.tight_layout()
save_figure(fig,'Fig18_raw_simba_cosine_disagreement_context')
plt.show()

### Interpretation and manual confirmation checklist

These examples are selected to illustrate a **discordance** between learned SIMBA similarity and modified cosine on the same experimental raw spectra. For manuscript use, inspect the exact adduct and charge, precursor isotope envelopes, chromatographic separation, common and shifted fragmentation patterns, and whether abundance is consistently higher in Cilni+Bac than both controls. The figure displays **hypothesized mass-shift relationships only**; a precursor-mass difference by itself is not evidence of a particular structural transformation. Report the selection thresholds and number of candidates passing them alongside the three examples to avoid overstating representativeness.

## 18C. **Publication figures:** D1, D2, D3, D4+, disconnected

The stacked bar plots compare **the top-K features identified by SIMBA** against **the top-K from modified cosine**, using identical candidate counts and the **new raw modified-cosine network**. We also show the distribution **among biologically enriched retrieved nodes only**.

Do not combine disconnected and insufficient-spectra cases. Publication-ready PDF, SVG and 300-dpi PNG files are saved to the dedicated raw-network directory.

In [ ]:
RAW_NET_STATUS_ORDER=['D1','D2','D3','D4+','Disconnected','Insufficient peaks']
RAW_NET_COLORS={'D1':'#2A647E','D2':'#4894A4','D3':'#8BBFBC','D4+':'#B9D7D0',
                'Disconnected':'#E3A261','Insufficient peaks':'#A0A3A6'}
RAW_NET_METHODS=[('SIMBA','raw_simba_rank'),('Modified cosine','raw_cosine_rank')]
raw_net_rank_table=raw_net_pool[['feature_id','raw_simba_rank','raw_cosine_rank','bio_support_proxy']].copy()
raw_net_rank_table['feature_id']=raw_net_rank_table.feature_id.astype(str)
if 'drug_specific_triage' in raw_net_pool:
    raw_net_rank_table['drug_specific_triage']=raw_net_pool['drug_specific_triage'].fillna(False).astype(bool).values
raw_net_comparison=[]
for graph_name in raw_net_networks:
    byid=raw_net_status_table.loc[raw_net_status_table.graph.eq(graph_name)].set_index('feature_id')
    z=raw_net_rank_table.join(byid[['raw_network_status','hop_distance','raw_node_degree']],
                              on='feature_id',validate='one_to_one')
    if z.raw_network_status.isna().any():
        raise RuntimeError('Network and ranking candidate pools differ')
    for method,col in RAW_NET_METHODS:
        for k in RAW_NET_PLOT_KS:
            top=z.nsmallest(min(k,len(z)),col)
            for status in RAW_NET_STATUS_ORDER:
                q=top.loc[top.raw_network_status.eq(status)]
                raw_net_comparison.append({'graph':graph_name,'method':method,'k':k,
                    'network_status':status,'n':len(q),
                    'bio_enriched_n':int(q.bio_support_proxy.fillna(False).sum()),
                    'fraction':len(q)/len(top) if len(top) else np.nan})
raw_net_counts=pd.DataFrame(raw_net_comparison)
raw_net_counts.to_csv(RAW_NET_DIR/'raw_network_topk_status_counts.csv',index=False)


def _raw_net_save(fig,name):
    for ext in ('pdf','svg','png'):
        fig.savefig(RAW_NET_DIR/f'{name}.{ext}',dpi=300,bbox_inches='tight',facecolor='white')


def _raw_net_plot_stacked(graph_name,ks=(5,10,20),enriched_only=False):
    fig,axes=plt.subplots(1,len(ks),figsize=(4.6*len(ks),4.8),sharey=True)
    axes=np.atleast_1d(axes)
    for ax,k in zip(axes,ks):
        left=np.zeros(2,dtype=float)
        for cat in RAW_NET_STATUS_ORDER:
            sub=raw_net_counts.loc[(raw_net_counts.graph==graph_name)&
                (raw_net_counts.k==k)&(raw_net_counts.network_status==cat)]
            measure='bio_enriched_n' if enriched_only else 'n'
            nums=np.array([sub.loc[sub.method.eq(method),measure].sum()
                           for method,_ in RAW_NET_METHODS],dtype=float)
            bars=ax.barh([0,1],nums,left=left,height=0.6,color=RAW_NET_COLORS[cat],label=cat)
            for bar,v,offset in zip(bars,nums,left):
                if v>0:ax.text(offset+v/2,bar.get_y()+bar.get_height()/2,
                                str(int(v)),ha='center',va='center',fontsize=9,
                                color='black' if cat in ('D3','D4+','Insufficient peaks') else 'white')
            left+=nums
        ax.set_yticks([0,1],['SIMBA','Modified cosine'])
        ax.invert_yaxis()
        ax.set_xlabel('Biologically enriched features' if enriched_only else 'Retrieved features')
        ax.set_title(f'Top {k}')
        ax.set_xlim(0,k)
        ax.grid(axis='x',alpha=0.15)
    fig.suptitle(f'Distance from cilnidipine in raw modified-cosine network\n{graph_name}' +
                 (' — biologically enriched only' if enriched_only else ''),fontsize=12)
    handles,labels=axes[-1].get_legend_handles_labels()
    fig.legend(handles,labels,loc='lower center',bbox_to_anchor=(0.5,-0.08),ncol=3,frameon=False)
    fig.tight_layout()
    return fig

for graph_name in raw_net_networks:
    slug='mutual' if graph_name.startswith('Mutual') else 'unpruned'
    for enrich in (False,True):
        fig=_raw_net_plot_stacked(graph_name,(5,10,20),enriched_only=enrich)
        _raw_net_save(fig,f'Fig_raw_network_top5_10_20_{slug}' + ('_bio' if enrich else ''))
        plt.show()

# Explicit counts with D1/D2/D3/D4+/disconnected for reporting.
display(raw_net_counts.loc[(raw_net_counts.graph==f'Mutual top-{RAW_NET_MUTUAL_TOP_N}')&
                           (raw_net_counts.k.isin([5,10,20]))]
    .pivot_table(index=['method','k'],columns='network_status',values='n',aggfunc='sum',fill_value=0)
    .reindex(columns=RAW_NET_STATUS_ORDER))


## 18D. Network topology plot and candidate-level comparison

Plots the **cilnidipine component** from the raw cosine-derived network, highlights **SIMBA Top 20** and **cosine Top 20**, and separately reports how many retrieved features lie outside the component. Disconnected nodes are counted in the stacked bar plots rather than being misleadingly placed near the connected component. A feature-level CSV supports manual inspection of SIMBA-only discoveries.

In [ ]:
raw_net_primary_label=f'Mutual top-{RAW_NET_MUTUAL_TOP_N}'
raw_net_primary_graph=raw_net_networks[raw_net_primary_label]
raw_net_primary_status=(raw_net_status_table.loc[
    raw_net_status_table.graph.eq(raw_net_primary_label),
    ['feature_id','raw_network_status','hop_distance','raw_node_degree']])
raw_net_ranked=raw_net_rank_table.merge(raw_net_primary_status,on='feature_id',validate='one_to_one')
raw_net_ranked['simba_top20']=raw_net_ranked.raw_simba_rank.le(20)
raw_net_ranked['cosine_top20']=raw_net_ranked.raw_cosine_rank.le(20)
raw_net_ranked['simba_only_top20']=raw_net_ranked.simba_top20 & ~raw_net_ranked.cosine_top20
raw_net_ranked.to_csv(RAW_NET_DIR/'raw_network_candidate_rank_distances.csv',index=False)

component_nodes=nx.node_connected_component(raw_net_primary_graph,raw_net_parent_id)
comp=raw_net_primary_graph.subgraph(component_nodes).copy()
if len(comp)>800:
    # Breadth-first truncated view, always preserving first-degree neighbors.
    distances=nx.single_source_shortest_path_length(comp,raw_net_parent_id)
    keep=sorted(distances,key=lambda fid:(distances[fid],str(fid)))[:800]
    comp=comp.subgraph(keep).copy()

fig,ax=plt.subplots(figsize=(10.5,7.8))
pos=nx.spring_layout(comp,seed=12,weight='cosine')
nx.draw_networkx_edges(comp,pos,ax=ax,edge_color='#BBBBBB',width=0.55,alpha=0.4)
s_top=set(raw_net_ranked.loc[raw_net_ranked.simba_top20,'feature_id'])
c_top=set(raw_net_ranked.loc[raw_net_ranked.cosine_top20,'feature_id'])
for ids,col,label,sz in [
    ((set(comp)-s_top-c_top-{raw_net_parent_id}),'#B3BEC2','Other network features',21),
    ((s_top-c_top)&set(comp),'#2D7EA8','SIMBA Top 20 only',85),
    ((c_top-s_top)&set(comp),'#E19943','Cosine Top 20 only',85),
    ((s_top&c_top)&set(comp),'#7B6AA3','Both Top 20',95),
    ({raw_net_parent_id},'#BF3C3C','Cilnidipine query',220)]:
    ids=sorted(ids)
    if ids:
        nx.draw_networkx_nodes(comp,pos,nodelist=ids,ax=ax,node_color=col,node_size=sz,label=label,
                               edgecolors='white',linewidths=0.5)
labels={fid:fid for fid in ((s_top|c_top|{raw_net_parent_id}) & set(comp))}
nx.draw_networkx_labels(comp,pos,labels=labels,ax=ax,font_size=7)
ax.legend(loc='upper left',frameon=False,fontsize=8)
ax.set_title('Cilnidipine component — raw MS/MS modified-cosine network\n'
             'Disconnected Top-20 candidates quantified separately')
ax.axis('off');fig.tight_layout()
_raw_net_save(fig,'Fig_raw_network_cilnidipine_component_highlights');plt.show()

print('SIMBA top 20 status counts:')
display(raw_net_ranked.loc[raw_net_ranked.simba_top20,
    ['feature_id','raw_simba_rank','raw_cosine_rank','bio_support_proxy',
     'raw_network_status','hop_distance']].sort_values('raw_simba_rank'))
print('Outputs saved in:',RAW_NET_DIR)


### Interpretation for the manuscript

Describe the numbers as **topological positions in a molecular network reconstructed from individual raw MS/MS spectra using modified cosine**. State the cosine threshold, peak tolerance, minimum matches, and reciprocal top-neighbor rule. Compare the matched candidate sets at **Top 5, 10, and 20**, optionally show Top 50 and Top 100. Disconnected candidates are meaningful *topologically*; they are not automatically verified structural analogs. Confirm chemical plausibility separately using the raw mirror spectra, mass shifts, and the abiotic/DMSO abundance controls from preceding sections.

**Sensitivity checks:** The default raw network uses one selected raw spectrum per feature with mutual top-neighbor pruning. The unpruned threshold network reveals how sensitive connectivity is to the neighbor pruning. For a more comprehensive analysis, run separate raw networks for each replicate or test different network thresholds; do not silently treat this GNPS-like construction as identical to the original GNPS2 network.

## 19. Export cilnidipine raw MS/MS query and the SAME 10 SIMBA-prioritized candidates

The ten candidates were already selected in Section 17, with **0.30 ≤ pooled modified cosine < 0.70**, cosine rank worse than SIMBA rank, and valid raw spectra, sorted by SIMBA rank (not by how low cosine is). This section exports **exactly those ten** without re-ranking or re-selection, ensuring both case-study examples are included. The query and candidate precursor m/z values and Δm/z are saved to the CSV audit. Individual-scan scores may differ from the pooled selection scores.


In [ ]:
# MGF export: raw spectra only (no consensus/processed spectra).
from pathlib import Path
import csv
import numpy as np

MGF_EXPORT_DIR = FIG_DIR / 'raw_msms_mgf_exports'
MGF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MGF_QUERY_PATH = MGF_EXPORT_DIR / 'cilnidipine_query_raw.mgf'
MGF_CANDIDATES_PATH = MGF_EXPORT_DIR / 'simba_top10_low_modified_cosine_raw.mgf'
MGF_AUDIT_PATH = MGF_EXPORT_DIR / 'simba_top10_low_modified_cosine_audit.csv'
# Reuse the exact ten candidates selected before the two-example figure.
# This guarantees that the two case-study feature IDs occur in this export.
assert len(mgf_ranked) == MGF_N_CANDIDATES
assert mgf_ranked.feature_id.is_unique
assert set(ADDITIONAL_CASE_IDS).issubset(set(mgf_ranked.feature_id))
assert mgf_ranked.raw_cosine_median.ge(MGF_MIN_COSINE).all()
assert mgf_ranked.raw_cosine_median.lt(MGF_MAX_COSINE).all()
assert mgf_ranked.feature_id.isin(chosen_records).all()
print('Two case-study features included among the ten:', ADDITIONAL_CASE_IDS)

# Same injection as the first selected candidate; use its previously extracted
# real cilnidipine query scan, without mixing scans from different injections.
mgf_first_id = str(mgf_ranked.iloc[0].feature_id)
mgf_query_filename = chosen_records[mgf_first_id][0]
mgf_query = raw_query_by_file[mgf_query_filename]

def _write_raw_mgf_entry(handle, record, title, feature_id, source_file):
    mzs = np.asarray(record['mz'], dtype=float)
    intensities = np.asarray(record['intensity'], dtype=float)
    valid = np.isfinite(mzs) & np.isfinite(intensities) & (mzs > 0) & (intensities > 0)
    mzs, intensities = mzs[valid], intensities[valid]
    if not len(mzs):
        raise ValueError(f'No valid raw peaks for {title}')
    order = np.argsort(mzs)
    precursor = float(record['precursor_mz'])
    if not np.isfinite(precursor) or precursor <= 0:
        raise ValueError(f'Invalid precursor m/z for {title}')
    handle.write('BEGIN IONS\n')
    handle.write(f'TITLE={title}\n')
    handle.write(f'PEPMASS={precursor:.8f}\n')
    handle.write('CHARGE=1+\n')
    handle.write(f'RTINSECONDS={float(record["rt_min"])*60:.4f}\n')
    handle.write(f'FEATURE_ID={feature_id}\n')
    handle.write(f'SOURCE_FILE={source_file}\n')
    handle.write(f'SCAN_ID={record["scan_id"]}\n')
    for idx in order:
        handle.write(f'{mzs[idx]:.8f} {intensities[idx]:.8g}\n')
    handle.write('END IONS\n\n')

with MGF_QUERY_PATH.open('w', encoding='utf-8') as out:
    _write_raw_mgf_entry(out, mgf_query, 'cilnidipine_parent_raw',
                         PARENT_FEATURE_ID, mgf_query_filename)

mgf_audit = []
with MGF_CANDIDATES_PATH.open('w', encoding='utf-8') as out:
    for row in mgf_ranked.itertuples(index=False):
        fid = str(row.feature_id)
        source_file, raw_scan = chosen_records[fid]
        _write_raw_mgf_entry(out, raw_scan, f'feature_{fid}_SIMBA_rank_{int(row.raw_simba_rank)}',
                             fid, source_file)
        mgf_audit.append({
            'feature_id': fid,
            'raw_simba_rank': int(row.raw_simba_rank),
            'raw_cosine_rank': int(row.raw_cosine_rank),
            'raw_cosine_median': float(row.raw_cosine_median),
            'raw_mces_dist': float(row.raw_mces_dist),
            'source_file': source_file,
            'scan_id': raw_scan['scan_id'],
            'cilnidipine_precursor_mz': float(mgf_query['precursor_mz']),
            'candidate_precursor_mz': float(raw_scan['precursor_mz']),
            'delta_precursor_mz_da': float(raw_scan['precursor_mz']) - float(mgf_query['precursor_mz']),
            'precursor_mz': float(raw_scan['precursor_mz']),
            'rt_min': float(raw_scan['rt_min']),
            'n_raw_peaks': len(raw_scan['mz']),
        })

with MGF_AUDIT_PATH.open('w', newline='', encoding='utf-8') as out:
    writer = csv.DictWriter(out, fieldnames=list(mgf_audit[0]))
    writer.writeheader()
    writer.writerows(mgf_audit)

print('Query MGF:', MGF_QUERY_PATH, '(1 spectrum)')
print('Candidate MGF:', MGF_CANDIDATES_PATH, f'({len(mgf_audit)} spectra)')
print('Audit CSV:', MGF_AUDIT_PATH)
display(pd.DataFrame(mgf_audit))


## 20. Mirror plots of the 10 exported SIMBA candidates with 0.30 ≤ pooled cosine < 0.70

Compare the **exact raw query spectrum exported to MGF** (blue, above) against each of the **10 individual raw candidate spectra** (orange, below). Peak intensities are scaled separately for visualization only. Each panel displays feature ID, SIMBA rank, modified-cosine rank and score, **cilnidipine precursor m/z**, **candidate precursor m/z**, and **Δm/z (candidate minus cilnidipine)**. The titles use the *actual precursor masses of the plotted raw scans* for a directly interpretable mass difference. Both candidate cutoffs apply to pooled cosine; these mirror plots use one representative scan per feature.


In [ ]:
# Display mirror plots for the SAME 10 raw scans exported in Section 19.
import matplotlib.pyplot as plt
import numpy as np

MGF_MIRROR_PATH = MGF_EXPORT_DIR / 'Fig_raw_MS2_mirrors_SIMBA_top10_low_cosine.png'
MGF_MIRROR_PDF_PATH = MGF_EXPORT_DIR / 'Fig_raw_MS2_mirrors_SIMBA_top10_low_cosine.pdf'


def _mgf_plot_peaks(record):
    mz = np.asarray(record['mz'], dtype=float)
    intensity = np.asarray(record['intensity'], dtype=float)
    keep = np.isfinite(mz) & np.isfinite(intensity) & (mz > 0) & (intensity > 0)
    mz, intensity = mz[keep], intensity[keep]
    if not len(mz):
        raise ValueError('Raw spectrum has no valid peaks')
    return mz, 100.0 * intensity / intensity.max()

# Query scan is identical to the one exported in cilnidipine_query_raw.mgf.
query_mz, query_int = _mgf_plot_peaks(mgf_query)

fig, axes = plt.subplots(5, 2, figsize=(18, 20), sharex=False, sharey=True)
axes = axes.ravel()
for ax, row in zip(axes, mgf_ranked.itertuples(index=False)):
    fid = str(row.feature_id)
    source_file, candidate_scan = chosen_records[fid]
    cand_mz, cand_int = _mgf_plot_peaks(candidate_scan)
    # Use a common fragment-mass axis for each mirror.
    x_max = max(float(query_mz.max()), float(cand_mz.max())) * 1.02
    ax.vlines(query_mz, 0, query_int, color='#2878A5', lw=0.8, alpha=0.85)
    ax.vlines(cand_mz, 0, -cand_int, color='#D87B32', lw=0.8, alpha=0.85)
    ax.axhline(0, color='#444444', lw=0.7)
    ax.set_xlim(0, x_max)
    ax.set_ylim(-110, 110)
    ax.set_yticks([-100, -50, 0, 50, 100])
    ax.set_yticklabels(['100', '50', '0', '50', '100'])
    ax.set_title(
        f'Feature {fid} | SIMBA #{int(row.raw_simba_rank)} | '
        f'mod. cosine #{int(row.raw_cosine_rank)} | mod cosine score {float(row.raw_cosine_median):.3f}\n'
        f'Cilnidipine m/z {float(mgf_query["precursor_mz"]):.4f}  |  '
        f'candidate m/z {float(candidate_scan["precursor_mz"]):.4f}  |  '
        f'Δm/z {float(candidate_scan["precursor_mz"])-float(mgf_query["precursor_mz"]):+.4f} Da',
        fontsize=10,
    )
    ax.set_xlabel('Fragment m/z')
    ax.set_ylabel('Relative intensity (%)')
    ax.grid(axis='x', alpha=0.15)

fig.suptitle(f'Raw MS/MS mirrors: cilnidipine (blue) vs. SIMBA top-10 candidates with {LOW_COSINE_MIN} ≤  modified cosine < 0.7 (orange)',
             fontsize=14, y=1.005)
fig.tight_layout()
fig.savefig(MGF_MIRROR_PATH, dpi=220, bbox_inches='tight')
fig.savefig(MGF_MIRROR_PDF_PATH, bbox_inches='tight')
plt.show()
print('Mirror plots saved:', MGF_MIRROR_PATH, 'and', MGF_MIRROR_PDF_PATH)
